In [ ]:
%pip install jinja2


In [ ]:
import pandas as pd,glob
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt


In [ ]:
df = pd.read_csv("../eabl parsed csv/sales_products_merged_cleaned.csv")
df["quantity_cases"] = df["quantity"] * df["Cases_Conversion"]
df["returns_cases"] = df["return_quantity"] * df["Cases_Conversion"]
df

In [ ]:
# Verification: confirm no within-SKU unit mixing
uom_per_product = df.groupby('Product_Code')['Default_UOM'].nunique()
print("SKUs using exactly 1 UOM throughout their history:", (uom_per_product == 1).sum(), "of", len(uom_per_product))
# Expected: 222 of 222 — confirms quantity needs no unit conversion for per-SKU forecasting

# Verification: confirm "ambiguous duplicates" are real transactions, not true dupes
key_cols = ['customer_code', 'visit_date', 'product', 'quantity']
amb = df[df.duplicated(subset=key_cols, keep=False)]
print("Ambiguous rows:", len(amb), "— check what differs:")
for c in ['salesperson', 'return_quantity', 'segment']:
    print(f"  {c}: differs in {amb.groupby(key_cols)[c].transform('nunique').gt(1).sum()} rows")
# If differences are concentrated in salesperson/return_quantity, these are real rows — do not drop

# The one real open item: how much does the opening 26-day gap affect your earliest weeks?
df['visit_date'] = pd.to_datetime(df['visit_date'])
weekly_totals = df.set_index('visit_date').resample('W-MON')['quantity'].sum()
print("\nFirst 4 weeks of data (check for an obvious undercount vs. later weeks):")
print(weekly_totals.head(4))
print("\nMedian of weeks 5 onward, for comparison:")
print(weekly_totals.iloc[4:].median())

In [ ]:
# Just the column names
print(list(df.columns))

In [ ]:
# Column names + data types + non-null counts
print(df.info())


In [ ]:
# Column names + data types, null counts, and unique value counts together
summary = pd.DataFrame({
    "dtype": df.dtypes,
    "non_null": df.notna().sum(),
    "nulls": df.isna().sum(),
    "unique_values": df.nunique()
})
print(summary)

In [ ]:
# Are they identical everywhere?
are_identical = (df["segment"] == df["segment_grouped"]).all()
print("Identical in every row?", are_identical)

# How many rows differ?
diff_count = (df["segment"] != df["segment_grouped"]).sum()
print("Rows where they differ:", diff_count)
print("Rows where they match:", len(df) - diff_count)

# See what the differences actually look like
diff_rows = df[df["segment"] != df["segment_grouped"]][["segment", "segment_grouped"]].drop_duplicates()
print(diff_rows)


### What is the finest level of detail your data is captured at, and is that level actually usable for building a model — or is it too sparse/noisy to model directly?

### Is there a trend in a specific customer purchasing eg Tusker in terms of quantity in different days of the week so that we can forecast what they are likely to but when they show up.

#### How many times a combination of "customer_code", "Product_Code", "visit_date" shows up

In [ ]:
df["visit_date"] = pd.to_datetime(df["visit_date"])
grp = df.groupby(["customer_code", "Product_Code", "visit_date"]).size()
grp

#### Counts how many buckets exist in total (i.e., how many distinct customer+product+day combinations show up at all).

In [ ]:
n_total = len(grp)
n_total

In [ ]:
#Counts how many buckets exist in total (i.e., how many distinct customer+product+day combinations show up at all).
pct_singleton = (grp == 1).mean() * 100
print(n_total, pct_singleton)


### Fixing of the CustomerMaster.xlsx dataset by removing useless Columns

In [ ]:
cm = pd.read_excel("../EABL dataset/CustomerMaster_cleanV1.xlsx")

cm

In [ ]:
cm.isna().sum()

In [ ]:
columns_to_drop = [
    "days_since_last_login",
    "GPS Co-ordinate",
    "RSS Last Login Date",
    "RSS Date Registed",
    "RSS First Login Date",
    "has_alternate_name",
    "price_group_missing",
    "rss_date_filled",
    "rss_date_unresolved",
    "registration_batch",
    "ever_logged_in",
    "login_status",
    "gps_missing",
    "gps_address_matched",
    "gps_unresolved",
]


# errors="ignore" means it won't crash if a name doesn't match exactly
cm_trimmed = cm.drop(columns=columns_to_drop, errors="ignore")

# Save to a NEW file — CustomerMaster.xlsx itself is never touched
cm_trimmed.to_excel("../EABL dataset/CustomerMaster_clean_forecast.xlsx", index=False)

print("Original columns:", len(cm.columns))
print("Trimmed columns:", len(cm_trimmed.columns))
print(list(cm_trimmed.columns))
cm_trimmed

In [ ]:
cm_trimmed.isna().sum()

### "Does my data actually have layers/Hierachy — like a family tree — or is everything just at one flat level pretending to have layers?"



In [ ]:
pm = pd.read_csv("../eabl parsed csv/product_master_cleaned.csv")     # columns: Category2, Brand, Product_Code

for c in ["Category2", "Brand"]:
#how many different Categories exist
#how many different Brands exist
    print(c, pm[c].nunique())
print("SKU", pm["Product_Code"].nunique())
for c in ["Division", "Area", "Territory", "Route"]:
    print(c, cm_trimmed[c].nunique())


### Determine if :
#### i) Using sales_products_merged_cleaned.csv over fact_table.csv was a right decsion
#### ii) If the Category and Category2  column are the same in the product_master_cleaned.csv dataset
#### iii) If the segment and segment_grouped  column are the same in the product_master_cleaned.csv dataset

In [ ]:
pm.isna().sum()

**Practical action:**

- Use `segment_grouped` as the default hierarchy level, since it offers more stability.
- If Restaurant/Supermarket-specific decisions ever matter to the business, keep the raw `segment` column available as a secondary cut to drill into.
- Don't discard the raw `segment` column entirely just because `segment_grouped` is the main tree.

In [ ]:
fact = pd.read_csv("../eabl parsed csv/fact_table.csv")                          # columns: Brand

pct_unmatched = fact["Brand"].isna().mean() * 100
pct_cat_diff = (pm["Category"] != pm["Category2"]).mean() * 100
pct_seg_diff = (df["segment"] != df["segment_grouped"]).mean() * 100
print(pct_unmatched, pct_cat_diff, pct_seg_diff)


## Does Any of My Dataset Contain "holiday", "payday", "pay day", "festive", "event"]

In [ ]:
keywords = ["holiday", "payday", "pay day", "festive", "event"]
for f in glob.glob("**/*.csv", recursive=True) + glob.glob("**/*.xlsx", recursive=True):
    try:
        cols = (pd.read_csv(f, nrows=0) if f.endswith(".csv") else pd.read_excel(f, nrows=0)).columns
    except Exception:
        continue
    hits = [c for c in cols if any(k in str(c).lower() for k in keywords)]
    if hits:
        print(f, hits)


## Seasonal Adjustment — Confidence by Period

**July–February (including December):**

- Any seasonal adjustment used for these months should be treated as an **assumption or industry benchmark**, not something proven by our own data.
- Don't let a model "discover" a seasonal pattern from our own data for these months — there's only one observation each, so any pattern found could just as easily be a one-off event as a real trend.
- Instead:
  - Use a conservative, documented assumption (e.g., an industry-standard "festive season uplift %" for beverage/alcohol distribution, or input from EABL's own commercial/sales team, who already have a sense of how demand shifts around Christmas).
  - Label it clearly as an **assumption**, not a fitted result — so nobody later mistakes it for something the data proved.

**March–June:**

- Can be treated with slightly more (though still limited) confidence.
- We can cautiously use our own data's pattern, but shouldn't treat it as fully confirmed — 2 occurrences is barely above the minimum to even compare, not enough to rule out coincidence.
- Use it, but keep it as a **soft signal** rather than a hard-coded rule.
### Using the March–June window to find holidays 

- Within the March–June window, several real Kenyan public holidays fall in **both 2025 and 2026** — so for these specific dates, there are genuinely 2 occurrences to compare, not just a vague "the whole month repeated" argument.

| Holiday | 2025 date | 2026 date |
|---|---|---|
| Good Friday | March 28 | April 3 |
| Easter Monday | March 31 | April 6 |
| Labour Day | May 1 | May 1 |
| Madaraka Day | June 1 | June 1 |

- Since each of these falls within the covered data range in both years, it's possible to check: did sales spike around Easter 2025 the same way they did around Easter 2026?
- That's a real, if modest, test — not just an assumption.

### How to apply it to a brand/region

- This connects to something already confirmed: the data has enough density at the Category × Segment intersection (only 1.1% of cells were thin) to detect uneven effects.
- So a reasonable question becomes: *"Did Spirits sales in Nightclubs spike more than Beer sales in Wholesalers around Labour Day, in both 2025 and 2026?"*
- That's a legitimate, testable question with the current data.

### Two things to do before actually running this

- **Build the calendar.** No holiday-marking table exists anywhere in the project — a small lookup needs to be created manually with these specific dates (plus any others relevant to March–June) and joined onto the date column.
- **Keep the confidence level honest.** Even with 2 occurrences, this is still "weak but usable" evidence, not proof — good enough to say *"this looks like a real pattern worth watching,"* not strong enough to bet a large inventory decision on without also applying business judgment.

> Good instinct — this is a much sharper way to use the March–June advantage than just treating the whole 4-month block as one vague area of "slightly more confidence."

In [ ]:
df["visit_date"] = pd.to_datetime(df["visit_date"])
months = sorted(df["visit_date"].dt.to_period("M").astype(str).unique())
name_counts = Counter(pd.Period(m).strftime("%B") for m in months)
print(months)
print(name_counts)


In [ ]:
df.shape

In [ ]:
pm = pd.read_csv("../eabl parsed csv/product_master_cleaned.csv")[["Product_Code", "Category2"]]

# join Category2 onto the sales data, matching on Product_Code
df = df.drop(columns=["Category2"], errors="ignore").merge(pm, on="Product_Code", how="left")   # file now has Category2 - replace it with the master's, by code

df["visit_date"] = pd.to_datetime(df["visit_date"])
df["month"] = df["visit_date"].dt.to_period("M")

cells = df.groupby(["Category2", "segment_grouped", "month"]).size()
print(len(cells), (cells < 30).mean() * 100, (cells < 5).mean() * 100)
print(cells.describe())

df.to_csv("../eabl parsed csv/sales_products_merged_cleaned_2newcategories.csv", index=False)
print(df.shape)

In [ ]:
df.shape

## Step 1 — Transaction error check (FPP 2nd ed., Section 12.9 "Outliers")

**What:** Every transaction is checked for values that are impossible (negative sales, missing dates)
or suspicious (very large orders, returns bigger than the sale, dates outside their report file's window, Sunday sales).

**Rule followed:** The book warns that outliers "may be errors, or they may simply be unusual",
and that replacing them without understanding why is dangerous. So this step **only flags** rows —
nothing is removed here. Each flagged row is reviewed with the business (Step 10) and the decision
applied in Step 11.

**Input:** `df` = `sales_products_merged_cleaned.csv` (all rows, before the 1 April cut in the next cell).
**Output:** `../eabl parsed csv/transaction_error_flags.csv`. `df` itself is not changed.


In [ ]:
# =====================================================================
# STEP 1 — Transaction error check (FPP 2nd ed., Section 12.9)
# Flag only — nothing is removed. Decisions are made at Step 10/11.
# =====================================================================
import re

chk = df.copy()                      # work on a copy so df stays untouched
chk["row_id"] = chk.index            # lets us find each flagged row again later
reasons = {}                         # reason -> (mask of flagged rows, suggested action)

def add_flag(mask, reason, action):
    reasons[reason] = (mask.fillna(False), action)
    print(f"{reason:<55} {int(mask.fillna(False).sum()):>7,} rows")

print("Rows checked:", f"{len(chk):,}\n")

# --- A. Quantity checks -----------------------------------------------
add_flag(chk["quantity"] < 0,
         "negative quantity", "remove - a sale cannot be negative")
add_flag((chk["quantity"] == 0) & (chk["return_quantity"] > 0),
         "zero quantity, returns only", "keep - return-only visit, adds 0 to sales")
add_flag((chk["quantity"] == 0) & (chk["return_quantity"] == 0),
         "zero quantity and zero returns", "remove - empty line")

# --- B. Return checks -------------------------------------------------
add_flag(chk["return_quantity"] < 0,
         "negative return", "remove or correct - a return cannot be negative")
add_flag((chk["return_quantity"] > chk["quantity"]) & (chk["quantity"] > 0),
         "returns larger than quantity sold", "confirm - may be returns from an earlier delivery")

# --- C. Unit conversion check -----------------------------------------
add_flag(chk["Cases_Conversion"].isna() | (chk["Cases_Conversion"] <= 0),
         "missing or zero case conversion", "fix in product master")

# --- D. Very large orders (per product, not one rule for all) --------
# "Usual top" = that SKU's 99th percentile order size in cases.
sku_p99 = chk.groupby("Product_Code")["quantity_cases"].quantile(0.99)
chk["sku_p99_cases"] = chk["Product_Code"].map(sku_p99)
add_flag(chk["quantity_cases"] > 5 * chk["sku_p99_cases"],
         "very large order (>5x the SKU's 99th percentile)", "ask business - real bulk order or typing error?")

# --- E. Date checks ---------------------------------------------------
add_flag(chk["visit_date"].isna(),
         "missing date", "remove")
add_flag((chk["visit_date"] < pd.Timestamp("2025-03-01")) | (chk["visit_date"] > pd.Timestamp("2026-07-11")),
         "date outside the data period", "remove or correct")

# Each report file name ends with its own window: DDMMYYYY(start) + DDMMYYYY(end).
# Two names are irregular: '...1105202531062025' (31 June does not exist -> use month end)
# and '...162202628022026' (start written as 1622026 = 16/2/2026).
def file_window(name):
    digits = re.search(r"_(\d+)\.xlsx$", name).group(1)
    start_txt, end_txt = digits[:-8], digits[-8:]
    end = pd.to_datetime(end_txt, format="%d%m%Y", errors="coerce")
    if pd.isna(end):
        end = pd.Timestamp(year=int(end_txt[4:]), month=int(end_txt[2:4]), day=1) + pd.offsets.MonthEnd(0)
    start = pd.Timestamp(year=int(start_txt[-4:]), month=int(start_txt[2:-4]), day=int(start_txt[:2]))
    return start, end

windows = {f: file_window(f) for f in chk["source_file"].unique()}
chk["file_start"] = chk["source_file"].map(lambda f: windows[f][0])
chk["file_end"]   = chk["source_file"].map(lambda f: windows[f][1])
add_flag((chk["visit_date"] < chk["file_start"]) | (chk["visit_date"] > chk["file_end"]),
         "date outside its report file's window", "investigate - possible date typing error")

add_flag(chk["visit_date"].dt.dayofweek == 6,
         "Sunday sale (very rare day)", "keep unless business says no Sunday trading")

# --- Build the flag list ----------------------------------------------
keep_cols = ["row_id", "source_file", "customer_code", "visit_date", "Product_Code", "product",
             "quantity", "return_quantity", "quantity_cases", "sku_p99_cases"]
parts = []
for reason, (mask, action) in reasons.items():
    hit = chk.loc[mask, keep_cols].copy()
    hit["reason"] = reason
    hit["suggested_action"] = action
    parts.append(hit)

flags = pd.concat(parts, ignore_index=True)
flags["in_forecast_period"] = flags["visit_date"] >= pd.Timestamp("2025-04-01")  # March rows are cut in the next cell anyway
flags["decision"] = "pending review"                                               # filled in at Step 10/11

flags.to_csv("../eabl parsed csv/transaction_error_flags.csv", index=False)

# --- Summary ----------------------------------------------------------
summary = (flags.groupby(["reason", "suggested_action"])
                .agg(rows=("row_id", "size"), in_forecast_period=("in_forecast_period", "sum"))
                .reset_index())
print("\n", summary.to_string(index=False))
print(f"\nDistinct transactions flagged: {flags['row_id'].nunique():,} of {len(chk):,}")
print("df unchanged:", df.shape)    # should still be (984292, 40), same as cell 31


In [ ]:
# ---- Step 1 triage: are the very large orders normal FOR THAT CUSTOMER? ----
# A big wholesaler buying 400 cases is extreme vs. all customers, but normal for itself.
# Rule: if the same customer bought the same SKU at >= half this size on ANOTHER date,
# the order is "normal for this customer" (explained). Otherwise it is unexplained.

big = flags[flags["reason"].str.startswith("very large")].copy()

cust_sku = df[["customer_code", "Product_Code", "visit_date", "quantity_cases"]]
def comparable_before(row):
    other = cust_sku[(cust_sku["customer_code"] == row["customer_code"]) &
                     (cust_sku["Product_Code"] == row["Product_Code"]) &
                     (cust_sku.index != row["row_id"])]
    return (other["quantity_cases"] >= 0.5 * row["quantity_cases"]).any()

big["normal_for_customer"] = big.apply(comparable_before, axis=1)

flags.loc[big.index, "suggested_action"] = big["normal_for_customer"].map({
    True:  "keep - normal size for this customer (bulk buyer)",
    False: "ask business - out of character even for this customer",
})

# Zero-quantity return-only rows: decided, no business input needed
flags.loc[flags["reason"] == "zero quantity, returns only", "decision"] = "keep - adds 0 to sales"

flags.to_csv("../eabl parsed csv/transaction_error_flags.csv", index=False)

print("Very large orders:", len(big))
print("  normal for that customer:", big["normal_for_customer"].sum())
print("  out of character (ask EABL):", (~big["normal_for_customer"]).sum())
print("\ndf unchanged:", df.shape)


In [ ]:
# ==========================================================================
# STEP 1 - Inspect the 13 "out of character" large orders (read-only)
# Uses big, df from the triage cell above. Nothing is changed or saved.
# ==========================================================================
odd = big[~big["normal_for_customer"]].copy()

rows = []
for _, r in odd.iterrows():
    hist = df[(df["customer_code"] == r["customer_code"]) & (df["Product_Code"] == r["Product_Code"])]
    other = hist[hist.index != r["row_id"]]["quantity_cases"]
    same_day = hist[hist["visit_date"] == r["visit_date"]]
    line = df.loc[r["row_id"]]
    rows.append({
        "Date":                    r["visit_date"].strftime("%d %b %Y (%a)"),
        "Customer":                r["customer_code"],
        "Segment":                 line["segment"],
        "Salesperson":             line["salesperson"],
        "Product":                 r["product"],
        "Product code":            r["Product_Code"],
        "This order (cases)":      round(r["quantity_cases"], 2),
        "Units typed (quantity)":  line["quantity"],
        "Other orders of it":      len(other),
        "Their usual order":       round(other.median(), 2) if len(other) else None,
        "Their biggest other":     round(other.max(), 2) if len(other) else None,
        "Times bigger than usual": round(r["quantity_cases"] / other.median(), 1) if len(other) and other.median() > 0 else None,
        "Product's 99th pct":      round(r["sku_p99_cases"], 2),
        "Lines that day":          len(same_day),
        "Looks like 10x typo?":    "yes" if len(other) and other.median() > 0
                                          and 7 <= r["quantity_cases"] / other.median() <= 13 else "",
    })

odd_table = pd.DataFrame(rows).sort_values("This order (cases)", ascending=False).reset_index(drop=True)
odd_table.index += 1
print(f"Out-of-character large orders: {len(odd_table)}   (customers: {odd['customer_code'].nunique()})")
with pd.option_context("display.max_columns", None, "display.width", 250):
    display(odd_table)

print("\nBy customer:")
display(odd_table.groupby(["Customer", "Segment"]).size().rename("orders").to_frame())

print("df unchanged:", df.shape)


In [ ]:
# ---- Step 1 follow-up: Sunday rows (read-only — uses df from cell 1, does not change it) ----
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
day_name = df["visit_date"].dt.day_name()          # kept separate, NOT added to df

# ---- 1. Sunday rows: how many, which dates, which salespeople ----
sunday_rows = df[day_name == "Sunday"]
print("Sunday rows (product lines):", len(sunday_rows))
print("Different Sundays with sales:", sunday_rows["visit_date"].nunique())
print("Customers involved:", sunday_rows["customer_code"].nunique())

print("\nSunday rows per date, salesperson and customer:")
print(sunday_rows.groupby([sunday_rows["visit_date"].dt.date, "salesperson", "customer_code"])
                 .size().rename("rows").to_string())

# ---- 2. Do the Sunday salespeople also sell on other days? ----
sunday_people = sunday_rows["salesperson"].unique()
is_sunday_person = df["salesperson"].isin(sunday_people)

week_table = (df[is_sunday_person]
                .groupby(["salesperson", day_name[is_sunday_person]]).size()
                .unstack(fill_value=0)
                .reindex(columns=day_order, fill_value=0))
week_table["Total rows"] = week_table.sum(axis=1)
week_table["Dates worked"] = df[is_sunday_person].groupby("salesperson")["visit_date"].nunique()

print("\nRows per day of the week for the Sunday salespeople:")
print(week_table.to_string())

for person in sunday_people:
    other_days = week_table.loc[person, day_order[:6]].sum()
    verdict = "YES - also sells Monday to Saturday" if other_days > 0 else "NO - only appears on Sundays"
    print(f"\n{person}: {verdict} ({other_days:,} rows on other days, "
          f"{week_table.loc[person, 'Sunday']:,} on Sundays)")

print("\ndf unchanged:", df.shape)   # should still be (984292, 40)


In [ ]:
# ---- Which month did most Sunday rows fall in? (read-only, uses df) ----
sunday = df[df["visit_date"].dt.dayofweek == 6]                       # 6 = Sunday

by_month = (sunday.groupby(sunday["visit_date"].dt.to_period("M"))
                  .agg(Sundays=("visit_date", "nunique"),
                       Rows=("visit_date", "size"),
                       Cases=("quantity_cases", "sum"),
                       Customers=("customer_code", "nunique")))
by_month["% of rows"]  = (by_month["Rows"]  / by_month["Rows"].sum()  * 100).round(1)
by_month["% of cases"] = (by_month["Cases"] / by_month["Cases"].sum() * 100).round(1)
by_month["Cases"] = by_month["Cases"].round(1)
by_month.index = by_month.index.strftime("%B %Y")                     # e.g. "February 2026"
by_month.index.name = "Month"

display(by_month)

top_rows  = by_month["Rows"].idxmax()
top_cases = by_month["Cases"].idxmax()
print(f"Most Sunday ROWS:  {top_rows}  ({by_month.loc[top_rows, 'Rows']} of {len(sunday)} rows, "
      f"{by_month.loc[top_rows, '% of rows']}%)")
print(f"Most Sunday CASES: {top_cases}  ({by_month.loc[top_cases, 'Cases']:,.0f} cases, "
      f"{by_month.loc[top_cases, '% of cases']}%)")
print("df unchanged:", df.shape)


In [ ]:
# ==========================================================================
# How did the Sunday rows affect sales? Week by week, and product by product (read-only, uses df)
# A week is a "spike" if it is above: median week + 3 x 1.4826 x MAD  (robust outlier rule)
# ==========================================================================
fc = df[df["visit_date"] >= pd.Timestamp("2025-04-01")]               # same period as the forecast data
week = fc["visit_date"].dt.to_period("W").dt.start_time               # Monday of each week, as in Prep cell 21
is_sun = fc["visit_date"].dt.dayofweek == 6
spike_rule = lambda s: s.median() + 3 * 1.4826 * (s - s.median()).abs().median()

# --------------------------------------------------------------------------
# PART A - PRODUCT-WEEKS: every product-week that contains Sunday rows
# --------------------------------------------------------------------------
weekly     = fc.groupby(["Product_Code", week])["quantity_cases"].sum()            # every product-week
weekly_sun = fc[is_sun].groupby(["Product_Code", week[is_sun]])["quantity_cases"].sum()
rows_sun   = fc[is_sun].groupby(["Product_Code", week[is_sun]]).size()

median_week = weekly.groupby(level=0).median()                        # each product's normal week
threshold_p = weekly.groupby(level=0).apply(spike_rule)               # each product's spike line

t = pd.DataFrame({"Sunday rows": rows_sun, "That week": weekly.loc[weekly_sun.index], "Sunday part": weekly_sun})
t["Without Sunday"] = t["That week"] - t["Sunday part"]
codes = t.index.get_level_values(0)
t["Normal week"] = codes.map(median_week).values
thr = codes.map(threshold_p).values
t["× normal (with)"]    = t["That week"] / t["Normal week"]
t["× normal (without)"] = t["Without Sunday"] / t["Normal week"]
t["Spike with Sunday"]    = t["That week"] > thr
t["Spike without Sunday"] = t["Without Sunday"] > thr
t["Effect of Sunday"] = np.select(
    [t["Spike with Sunday"] & ~t["Spike without Sunday"], t["Spike with Sunday"] & t["Spike without Sunday"]],
    ["CREATED a spike", "Spike anyway (week was high)"], default="No spike - normal week")

# --------------------------------------------------------------------------
# PART B - CALENDAR WEEKS: the weeks that contain a Sunday sale (all products together)
# --------------------------------------------------------------------------
total_week  = fc.groupby(week)["quantity_cases"].sum()                # company total per week
thr_total   = spike_rule(total_week)
weeks = sorted(week[is_sun].unique())

wk = pd.DataFrame(index=pd.DatetimeIndex(weeks, name="Week starting"))
wk["Week (Mon–Sun)"]   = [f"{d:%d %b} – {(d + pd.Timedelta(days=6)):%d %b %Y}" for d in wk.index]
wk["Sunday"]           = [f"{(d + pd.Timedelta(days=6)):%a %d %b %Y}" for d in wk.index]
wk["Sunday rows"]      = rows_sun.groupby(level=1).sum().reindex(wk.index).values
wk["Product-weeks"]    = rows_sun.groupby(level=1).size().reindex(wk.index).values
wk["Week total (cases)"]  = total_week.reindex(wk.index).values
wk["Sunday part (cases)"] = weekly_sun.groupby(level=1).sum().reindex(wk.index).values
wk["% of week from Sunday"] = wk["Sunday part (cases)"] / wk["Week total (cases)"] * 100
wk["Without Sunday (cases)"] = wk["Week total (cases)"] - wk["Sunday part (cases)"]
wk["× normal week (with)"]    = wk["Week total (cases)"] / total_week.median()
wk["× normal week (without)"] = wk["Without Sunday (cases)"] / total_week.median()
effects = t.groupby([t.index.get_level_values(1), "Effect of Sunday"]).size().unstack(fill_value=0)
for col in ["CREATED a spike", "Spike anyway (week was high)", "No spike - normal week"]:
    wk[f"Products: {col}"] = effects.reindex(wk.index)[col].fillna(0).astype(int).values if col in effects else 0
wk["Did Sunday affect this week?"] = np.where(
    wk["Products: CREATED a spike"] > 0,
    "YES - created spikes in " + wk["Products: CREATED a spike"].astype(str) + " product(s)",
    np.where(wk["Week total (cases)"] > thr_total, "Week is high overall, but Sunday not the cause",
             "NO - week stays normal"))

wk_show = wk.reset_index(drop=True)
wk_show.index += 1
for c in ["Week total (cases)", "Sunday part (cases)", "Without Sunday (cases)"]:
    wk_show[c] = wk_show[c].round(0).astype(int)
wk_show["% of week from Sunday"] = wk_show["% of week from Sunday"].round(1).astype(str) + "%"
for c in ["× normal week (with)", "× normal week (without)"]:
    wk_show[c] = wk_show[c].round(2).astype(str) + "×"

print(f"PART B - The {len(wk)} calendar weeks that contain Sunday sales "
      f"(company normal week = {total_week.median():,.0f} cases):")
with pd.option_context("display.max_columns", None, "display.width", None):
    display(wk_show.T)                                             # one column per week, easy to compare

# ---- Summary of product-weeks ----
summary_spikes = (t["Effect of Sunday"].value_counts()
                    .reindex(["CREATED a spike", "Spike anyway (week was high)", "No spike - normal week"], fill_value=0)
                    .rename("Product-weeks").to_frame())
summary_spikes["Sunday rows"] = t.groupby("Effect of Sunday")["Sunday rows"].sum().reindex(summary_spikes.index).fillna(0).astype(int)
summary_spikes.loc["Total"] = summary_spikes.sum()
print("\nPART A - Effect on the product-weeks:")
display(summary_spikes)

# ---- Every product-week, grouped by calendar week ----
names = df.drop_duplicates("Product_Code").set_index("Product_Code")["product"]
detail_sun = t.copy()
detail_sun.insert(0, "Product", detail_sun.index.get_level_values(0).map(names))
detail_sun.insert(1, "Week starting", detail_sun.index.get_level_values(1))
order = {"CREATED a spike": 0, "Spike anyway (week was high)": 1, "No spike - normal week": 2}
detail_sun = (detail_sun.assign(_o=detail_sun["Effect of Sunday"].map(order))
                        .sort_values(["Week starting", "_o", "Sunday part"], ascending=[True, True, False])
                        .drop(columns=["_o", "Spike with Sunday", "Spike without Sunday"])
                        .reset_index(drop=True))
for c in ["That week", "Sunday part", "Without Sunday", "Normal week"]:
    detail_sun[c] = detail_sun[c].round(0).astype(int)
for c in ["× normal (with)", "× normal (without)"]:
    detail_sun[c] = detail_sun[c].round(1).astype(str) + "×"

with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.width", None):
    for wk_start, g in detail_sun.groupby("Week starting"):
        print(f"\n=== Week {wk_start:%d %b} – {(wk_start + pd.Timedelta(days=6)):%d %b %Y}: "
              f"{len(g)} product-weeks, {g['Sunday rows'].sum()} Sunday rows ===")
        display(g.drop(columns="Week starting").reset_index(drop=True).rename(index=lambda i: i + 1))

print("df unchanged:", df.shape)   # should still be (984292, 40)


In [ ]:
# ==========================================================================
# STEP 9 (start) - Flag the known Sunday spike: week of 8 Dec 2025
# Inputs (must come from THESE cells, run just before this one):
#   t      -> Sunday-spike cell ("How did the Sunday rows affect sales?")
#   names  -> same Sunday-spike cell (Product_Code -> product name)
#   flags  -> Step 1 check cell, updated by the triage cell
# df is NOT changed. Writes: the Step 1 flags file (updated) + known_spikes.csv.
# ==========================================================================

# ---- Safety checks: stop if t / names / flags are not the ones we expect ----
assert {"Effect of Sunday", "Sunday part", "That week"}.issubset(t.columns), \
    "`t` is not the Sunday-spike table - re-run the Sunday-spike cell first"
assert names.index.name == "Product_Code", "`names` is not the product-name list - re-run the Sunday-spike cell"
assert (flags["reason"] == "Sunday sale (very rare day)").sum() == 95, \
    "`flags` is not the Step 1 flags table - re-run the Step 1 cell and the triage cell"

# ---- 1. List the high product-weeks that contain Sunday rows ----
sun_spike_weeks = t[t["Effect of Sunday"] != "No spike - normal week"].copy()
sun_spike_weeks.index.names = ["Product_Code", "week_start"]
sun_spike_weeks = sun_spike_weeks.reset_index()
sun_spike_weeks.insert(1, "product", sun_spike_weeks["Product_Code"].map(names))
sun_spike_weeks["status"] = np.where(
    sun_spike_weeks["Effect of Sunday"] == "CREATED a spike",
    "Sunday-caused spike - keep for now; with/without test (I2) + ask EABL about the 14 Dec order",
    "Week high even without Sunday - not caused by Sunday; review in outlier screen (F8)")
known_spikes = sun_spike_weeks[["Product_Code", "product", "week_start", "Sunday rows", "That week",
                                "Sunday part", "Without Sunday", "Normal week", "× normal (with)",
                                "× normal (without)", "Effect of Sunday", "status"]]

n_sun_created = (known_spikes["Effect of Sunday"] == "CREATED a spike").sum()
print(f"Product-weeks flagged: {len(known_spikes)}  "
      f"({n_sun_created} created by Sunday, {len(known_spikes) - n_sun_created} high anyway)")
with pd.option_context("display.max_columns", None, "display.width", 250):
    display(known_spikes.round(2))
known_spikes.to_csv("../eabl parsed csv/known_spikes.csv", index=False)

# ---- 2. Write the decision into the Step 1 flags for the 95 Sunday rows ----
sun_flag_rows = flags["reason"] == "Sunday sale (very rare day)"
sun_flag_key  = pd.Series(list(zip(flags["Product_Code"],
                                   flags["visit_date"].dt.to_period("W").dt.start_time)), index=flags.index)
is_created = known_spikes["Effect of Sunday"] == "CREATED a spike"
sun_created_keys = set(zip(known_spikes.loc[is_created, "Product_Code"],  known_spikes.loc[is_created, "week_start"]))
sun_anyway_keys  = set(zip(known_spikes.loc[~is_created, "Product_Code"], known_spikes.loc[~is_created, "week_start"]))

flags.loc[sun_flag_rows, "decision"] = "keep - key-account order, no effect on weekly sales"
flags.loc[sun_flag_rows & sun_flag_key.isin(sun_anyway_keys), "decision"] = \
    "keep - week was high without Sunday; check in outlier screen (F8)"
flags.loc[sun_flag_rows & sun_flag_key.isin(sun_created_keys), "decision"] = \
    "keep for now - Sunday-caused spike (week of 8 Dec); with/without test + EABL"
flags.to_csv("../eabl parsed csv/transaction_error_flags.csv", index=False)

print("\nDecisions now written for the 95 Sunday rows:")
display(flags.loc[sun_flag_rows, "decision"].value_counts().to_frame("rows"))
print("Flags rows (nothing removed):", len(flags), "| df unchanged:", df.shape)


In [ ]:
# ==========================================================================
# STEP 1 - Write the remaining decisions into the flags file (item H3)
#
# Where each input comes from (all already in memory - run these cells first):
#   flags -> built in the Step 1 check cell (cell 34, "STEP 1 — Transaction error check"):
#            one row per flagged transaction, with columns reason / suggested_action / decision.
#            Updated by the triage cell (cell 35) -> suggested_action for the 324 large orders
#            and by the Sunday-flag cell (cell 41) -> decision for the 95 Sunday rows.
#   df    -> the daily sales data loaded in cell 2 (only read here, for a final size check).
#   pd    -> pandas, imported at the top of the notebook.
#
# What it changes: ONLY the text in flags["decision"] for 328 rows. No rows removed.
# What it writes:  ../eabl parsed csv/transaction_error_flags.csv (the flags file from cell 34).
# ==========================================================================

# Exact reason texts written by cell 34 (copied from its add_flag(...) calls)
LG_REASON  = "very large order (>5x the SKU's 99th percentile)"
RET_REASON = "returns larger than quantity sold"

# ---- Safety checks: stop (and write nothing) if `flags` is not the expected table ----
assert (flags["reason"] == LG_REASON).sum() == 333, "`flags` is not the Step 1 table - re-run cell 34, then cell 35 (333 = 324 before July + 9)"
assert (flags["reason"] == RET_REASON).sum() == 4,  "`flags` is not the Step 1 table - re-run cell 34"
assert flags.loc[flags["reason"] == LG_REASON, "suggested_action"].str.startswith(
           ("keep - normal size", "ask business - out of character")).all(), \
    "the 324 large orders have no triage result yet - run the triage cell (cell 35) first"

# ---- Which rows get a decision ----
lg_order_rows = flags["reason"] == LG_REASON          # the 324 very large orders
ret_over_rows = flags["reason"] == RET_REASON         # the 4 returns > quantity rows

# Large orders: follow the triage result written by cell 35
flags.loc[lg_order_rows & flags["suggested_action"].str.startswith("keep"), "decision"] = \
    "keep - normal size for this customer (bulk buyer)"
flags.loc[lg_order_rows & flags["suggested_action"].str.startswith("ask"), "decision"] = \
    "keep for now - out of character; ask EABL, re-check in outlier screen (F8)"

# Returns > quantity: all 4 are Returnable empties (result of the check-5 cell)
flags.loc[ret_over_rows, "decision"] = \
    "keep - Returnable empties (2 split records, 2 empties from earlier purchases)"

flags.to_csv("../eabl parsed csv/transaction_error_flags.csv", index=False)

# ---- Show the result ----
print("Decisions by check:")
with pd.option_context("display.max_colwidth", None, "display.width", 250):
    display(flags.groupby(["reason", "decision"]).size().to_frame("rows"))
print("Still 'pending review':", (flags["decision"] == "pending review").sum(), "(should be 0)")
print("Flags rows (nothing removed):", len(flags), "| df unchanged:", df.shape)


### Step 1 – Decisions on the transaction checks
*FPP2 12.9: no unusual value is removed or replaced without a recorded reason ("simply replacing outliers without thinking about why they have occurred is a dangerous practice"). FPP2 1.6: unexplained values go to the people who know the data (EABL).*

**Result:** 501 flagged rows (496 transactions; 5 flagged for two reasons: Sunday sale + very large order). **Nothing removed.** Every row has a decision in `transaction_error_flags.csv` (0 left as "pending review"). Counts include the July 2026 data (to 11 Jul 2026).

| Check | Rows | Decision | Why |
|---|---|---|---|
| Negative quantity, negative return, zero-and-zero lines | 0 | – | Clean |
| Missing date, date outside data period, date outside its report file's window | 0 | – | Clean |
| Missing or zero case conversion | 0 | – | Clean |
| Zero quantity, returns only | 69 | Keep | All Returnable empties. Quantity is 0, so they add nothing to sales; the weekly sum (Prep) removes them anyway. |
| Returns larger than quantity sold | 4 | Keep | All Returnable: 2 split records (another line that day covers it), 2 empties from earlier purchases. |
| Very large orders (> 5 × the product's 99th percentile) | 333 | Keep 320 · Keep 13 for now and ask EABL | 332 of 333 by KEY ACCOUNT (Wholesaler 305, Specialist Store 20, Nightclub 4, Bar 3, Event 1). 320 match the customer's own past order sizes (normal bulk buying). 13 have no similar order from that customer; most come in matching drink + crate pairs (e.g. Balozi 500ml Ret + "Complete – Euros" crates, 141 + 141 cases on 6 Oct 2025), so likely real. New with July: Guinness (K) per bottle, 15 cases, 11 Jul 2026. Confirm with EABL; re-check in the outlier screen (F8). 4 of the 320 are dated before 7 Apr 2025 (1 in March, 3 on 1–6 Apr) and fall outside the forecast data after the start cuts (cell 50, D1). |
| Sunday sales (outlets usually closed) | 95 | Keep | On 4 Sundays: 30 Nov 2025 (1 row), 14 Dec 2025 (33), 1 Feb 2026 (40), 1 Mar 2026 (21); 94 of 95 by KEY ACCOUNT (which also works Mon–Sat, so not a batch of other days). 82 rows: no effect on weekly sales. **Week of 8 Dec 2025:** 7 rows created spikes for 5 products → with/without test in F1 + ask EABL about the 14 Dec order; 6 rows sit in weeks that were high anyway (3 products) → outlier screen (F8). Listed in `known_spikes.csv`. |

**Waiting on EABL:** the 13 out-of-character large orders (incl. the 15-case Guinness per-bottle order on 11 Jul 2026) and the 14 Dec 2025 Sunday order.
**Files:** `transaction_error_flags.csv` (every flagged row + decision; 501 rows), `known_spikes.csv` (8 spike product-weeks).
**After the Returnable split (Prep cell R1):** 101 of the 501 flagged rows are Returnable packaging (all 69 "returns only", all 4 "returns larger than quantity", and 28 of the 333 very large orders). Returnable is kept in this file and in `sales_products_forecast_ready.csv`, but the forecast (Prep onwards) uses EABL drinks only. So **8 of the 13 "ask EABL" large orders** affect the drinks forecast; the other 5 are crates/bottles. All 8 rows in `known_spikes.csv` are drinks.




In [ ]:
# ==========================================================================
# STEP 6 - PART 2b: RE-CHECK the very large orders with EARLIER DATES ONLY (leaks A and B)
#          FLAG ONLY - nothing in df or in transaction_error_flags.csv is changed
#
# WHAT WAS LEAKY (Step 1 cells above):
#   Leak A (Step 1 check cell, "very large order"): an order is "very large" if it is > 5 x that
#     product's 99th-percentile order size - but the percentile was worked out from ALL dates,
#     including the test months (Apr-Jul 2026). A later bulk order could raise the bar for 2025.
#   Leak B (triage cell, "normal for this customer"): a large order was excused if the same customer
#     bought >= half that size of the same product on ANY other date - including LATER dates.
#
# THE FIX (same rules, same numbers, only the dates change):
#   A: the 99th percentile uses only that product's order lines dated ON OR BEFORE the order's day
#      (never a later day). Needs at least 20 such lines, otherwise "too early to judge".
#   B: "normal for this customer" only counts the customer's OTHER lines of the same product dated
#      ON OR BEFORE the order's day (never a later day).
#   Same day is allowed: it is not the future (split records on one day are one visit).
#   Only orders up to Sat 11 Apr 2026 (end of week 53, the seal week of Mon 6 Apr 2026) are judged.
#   Later orders (test weeks 54-66 and after) are listed but NOT judged (they belong to Step 9).
#   Because every judgement only looks back, the result is also valid for CV round 1 (<= wk 27)
#   and round 2 (<= wk 40), like F8.
#
# WHY (FPP2, 2nd ed.):
#   12.9 (p.488): outliers "may be errors, or they may simply be unusual" and "Simply replacing
#     outliers without thinking about why they have occurred is a dangerous practice"
#     -> flag only; decisions stay with EABL (G1).
#   3.4 (p.84): "no future observations can be used in constructing the forecast" -> a check that
#     decides what training data looks like may not use later dates.
#   4.2 (p.105) "Document and justify" -> results saved to a NEW documented file.
#
# WHERE EACH INPUT COMES FROM (read from disk, READ ONLY - so no earlier cell has to be re-run):
#   ../eabl parsed csv/sales_products_merged_cleaned.csv -> the same file cell 2 loads into df
#            (one row per order line; its row number = row_id in the flags file, as in cell 34)
#   ../eabl parsed csv/transaction_error_flags.csv -> saved by cells 34, 35, 41, 42: one row per
#            flagged line (row_id, reason, suggested_action, decision)
#   quantity_cases = quantity x Cases_Conversion -> the same formula as cell 2
#   pd, np -> cell 1 (imports);  display -> Jupyter built-in
#   The notebook's own df and flags are NOT used and NOT changed (later cells change df).
#
# WHAT IT CHANGES: nothing. Both files are only read.
# WHAT IT WRITES:  ONE NEW file ../eabl parsed csv/large_order_flags_earlier_only.csv
#                  (transaction_error_flags.csv and sales_products_merged_cleaned.csv are NOT
#                  touched). If a file with that name exists and was NOT made by this cell
#                  (no made_by = "P2b" column), it stops and writes nothing.
# All new names start with p2b_ / P2B_ so nothing clashes with other cells.
# ==========================================================================
import os

P2B_LG_REASON = "very large order (>5x the SKU's 99th percentile)"   # exact text from cell 34
P2B_X, P2B_Q, P2B_HALF = 5, 0.99, 0.5            # the SAME numbers as cells 34 and 35
P2B_MIN_LINES = 20                                 # fewer earlier lines -> "too early to judge" (our choice)
P2B_SEAL_END = pd.Timestamp("2026-04-11")          # Saturday of week 53 (week of Mon 6 Apr 2026)
P2B_PATH = "../eabl parsed csv/large_order_flags_earlier_only.csv"
P2B_MADE_BY = "P2b Cleaning notebook (Step 6 Part 2b)"

# ---- Read the two input files (read only) ----
p2b_df = pd.read_csv("../eabl parsed csv/sales_products_merged_cleaned.csv",
                     usecols=["customer_code", "Product_Code", "product", "visit_date", "quantity", "Cases_Conversion"],
                     dtype={"Product_Code": str, "customer_code": str})
p2b_df["visit_date"] = pd.to_datetime(p2b_df["visit_date"])
p2b_df["quantity_cases"] = p2b_df["quantity"] * p2b_df["Cases_Conversion"]          # same formula as cell 2
p2b_flags = pd.read_csv("../eabl parsed csv/transaction_error_flags.csv", dtype={"Product_Code": str, "customer_code": str})

# ---- Safety checks: stop if the files are not the ones this cell expects ----
assert (p2b_flags["reason"] == P2B_LG_REASON).sum() == 333, "flags file is not the Step 1 table (333 large orders)"
p2b_old = p2b_flags[p2b_flags["reason"] == P2B_LG_REASON].set_index("row_id")
assert p2b_old.index.isin(p2b_df.index).all() and \
       (p2b_df.loc[p2b_old.index, "customer_code"].values == p2b_old["customer_code"].values).all() and \
       np.allclose(p2b_df.loc[p2b_old.index, "quantity_cases"].values, p2b_old["quantity_cases"].values), \
    "row_id in the flags file no longer matches the sales file rows - stop and check"

# ---- PROOF: the old all-dates rules, re-done here, must give exactly cells 34-35's results ----
p2b = p2b_df[["customer_code", "Product_Code", "visit_date", "quantity_cases"]].copy()
p2b["p99_all_dates"] = p2b["Product_Code"].map(p2b.groupby("Product_Code")["quantity_cases"].quantile(P2B_Q))
p2b["large_all_dates"] = p2b["quantity_cases"] > P2B_X * p2b["p99_all_dates"]
assert set(p2b.index[p2b["large_all_dates"]]) == set(p2b_old.index), "cannot reproduce cell 34's 333 large orders - stop"

# ---- Fix A: 99th percentile from lines ON OR BEFORE the order's day (never later) ----
p2b = p2b.sort_values(["Product_Code", "visit_date"], kind="stable")
p2b["p2b_n"] = p2b.groupby("Product_Code").cumcount() + 1
p2b["p2b_q"] = p2b.groupby("Product_Code")["quantity_cases"].transform(lambda s: s.expanding().quantile(P2B_Q))
p2b_day = p2b.groupby(["Product_Code", "visit_date"])[["p2b_q", "p2b_n"]].last()   # value at the END of each day
p2b = p2b.join(p2b_day.rename(columns={"p2b_q": "p99_earlier", "p2b_n": "lines_earlier"}),
               on=["Product_Code", "visit_date"]).drop(columns=["p2b_q", "p2b_n"]).sort_index()
p2b["judged"] = p2b["visit_date"] <= P2B_SEAL_END
p2b["large_earlier"] = p2b["judged"] & (p2b["lines_earlier"] >= P2B_MIN_LINES) & \
                       (p2b["quantity_cases"] > P2B_X * p2b["p99_earlier"])

# ---- Fix B: "normal for this customer" from the customer's OTHER lines ON OR BEFORE that day ----
p2b_cand = p2b[p2b["large_all_dates"] | p2b["large_earlier"]].copy()      # every order either rule calls large
p2b_cs = p2b_df[["customer_code", "Product_Code", "visit_date", "quantity_cases"]]
def p2b_normal(row, earlier_only):
    other = p2b_cs[(p2b_cs["customer_code"] == row["customer_code"]) & (p2b_cs["Product_Code"] == row["Product_Code"])
                   & (p2b_cs.index != row.name)]
    if earlier_only:
        other = other[other["visit_date"] <= row["visit_date"]]
    return bool((other["quantity_cases"] >= P2B_HALF * row["quantity_cases"]).any())
p2b_cand["normal_all_dates"] = p2b_cand.apply(p2b_normal, axis=1, earlier_only=False)
p2b_cand["normal_earlier"]   = p2b_cand.apply(p2b_normal, axis=1, earlier_only=True)
p2b_old_ok = p2b_old["suggested_action"].str.startswith("keep - normal size")
assert (p2b_cand.loc[p2b_old.index, "normal_all_dates"] == p2b_old_ok).all(), "cannot reproduce cell 35's triage - stop"

# ---- Result per order: old (all dates) vs new (earlier dates only) ----
def p2b_result(large, normal):
    return "not large" if not large else ("large - normal for customer" if normal else "large - ask EABL")
p2b_cand["result_all_dates"] = [p2b_result(l, n) for l, n in zip(p2b_cand["large_all_dates"], p2b_cand["normal_all_dates"])]
p2b_cand["result_earlier"] = [("after the seal - not judged (Step 9)" if not j else
                               "too early to judge" if (l_all and le < P2B_MIN_LINES) else p2b_result(l, n))
                              for j, l, n, l_all, le in zip(p2b_cand["judged"], p2b_cand["large_earlier"],
                                                            p2b_cand["normal_earlier"], p2b_cand["large_all_dates"],
                                                            p2b_cand["lines_earlier"])]
p2b_cand["changed"] = p2b_cand["judged"] & (p2b_cand["result_all_dates"] != p2b_cand["result_earlier"])
p2b_out = (p2b_cand.reset_index().rename(columns={"index": "row_id"})
           .merge(p2b_flags.loc[p2b_flags["reason"] == P2B_LG_REASON, ["row_id", "product", "decision"]]
                  .rename(columns={"decision": "old_decision"}), on="row_id", how="left"))
p2b_out["product"] = p2b_out["product"].fillna(p2b_out["row_id"].map(p2b_df["product"]))
p2b_out["old_decision"] = p2b_out["old_decision"].fillna("not in flags file (not large on all dates)")
p2b_out["made_by"] = P2B_MADE_BY
p2b_out = p2b_out[["row_id", "visit_date", "customer_code", "Product_Code", "product", "quantity_cases",
                   "p99_all_dates", "p99_earlier", "lines_earlier", "result_all_dates", "result_earlier",
                   "changed", "old_decision", "made_by"]].sort_values("visit_date").reset_index(drop=True)
p2b_out[["p99_all_dates", "p99_earlier"]] = p2b_out[["p99_all_dates", "p99_earlier"]].round(3)

# ---- Checks ----
assert p2b_out["row_id"].is_unique and set(p2b_old.index).issubset(set(p2b_out["row_id"]))
assert (p2b_out.loc[~p2b_out["visit_date"].le(P2B_SEAL_END), "result_earlier"] == "after the seal - not judged (Step 9)").all()

# ---- Never overwrite a file this cell did not make; write; read back ----
if os.path.exists(P2B_PATH):
    p2b_prev = pd.read_csv(P2B_PATH, nrows=1)
    assert "made_by" in p2b_prev.columns and str(p2b_prev["made_by"].iloc[0]).startswith("P2b"), \
        f"{P2B_PATH} exists and was NOT made by Part 2b. Nothing written; rename or check it."
    print(f"Note: {P2B_PATH} already existed (made by an earlier Part 2b run) - replaced with this run's version.")
p2b_out.to_csv(P2B_PATH, index=False)
p2b_back = pd.read_csv(P2B_PATH)
assert p2b_back.shape == p2b_out.shape and (p2b_back["row_id"].values == p2b_out["row_id"].values).all(), \
    "the saved file does not match the table - stop and check"

# ---- Report ----
p2b_j = p2b_out[p2b_out["visit_date"] <= P2B_SEAL_END]
print(f"Saved {P2B_PATH}: {len(p2b_out)} orders x {p2b_out.shape[1]} columns (read back - OK) | sales lines read: {len(p2b_df):,}")
print("Proof: the all-dates rules here reproduce cell 34 (333 large orders) and cell 35 (triage) exactly - OK\n")
print(f"Orders judged (up to {P2B_SEAL_END.date()}): {len(p2b_j)} | after the seal, not judged: {len(p2b_out) - len(p2b_j)}")
print("\nOld (all dates) vs new (earlier dates only), orders up to the end of week 53:")
display(pd.crosstab(p2b_j["result_all_dates"], p2b_j["result_earlier"], margins=True, margins_name="total"))
print(f"\nOrders whose result CHANGES: {int(p2b_j['changed'].sum())}")
with pd.option_context("display.max_colwidth", 45, "display.width", 250):
    display(p2b_j[p2b_j["changed"]][["visit_date", "customer_code", "product", "quantity_cases", "p99_all_dates",
                                     "p99_earlier", "result_all_dates", "result_earlier"]])
print("\nThe 13 'ask EABL' orders from cell 35 - what the earlier-only rule says:")
display(p2b_out[p2b_out["old_decision"].str.startswith("keep for now")]
        [["visit_date", "customer_code", "product", "quantity_cases", "result_earlier"]])


In [ ]:
# ---- Do reps record the SALE and the RETURN of empties on separate lines? (read-only) ----
cols = ["source_file", "salesperson", "customer_code", "visit_date", "product", "quantity", "return_quantity"]
lines = df[cols].copy()
lines["row_no"] = df.index                      # position of the row in the file (to check "the row beneath")

# The 68 "return-only" rows: nothing bought, empties handed back
ret_only = lines[(lines["quantity"] == 0) & (lines["return_quantity"] > 0)]
total = len(ret_only)
pct = lambda n: f"{n:,} of {total} ({n / total:.1%})"
print("Return-only rows:", total)

# ---- 1. Are there duplicates among the 68? ----
dup_mask = ret_only.duplicated(subset=["customer_code", "visit_date", "product", "quantity", "return_quantity"], keep=False)
print("\n1. Duplicates among the return-only rows:", pct(dup_mask.sum()))
if dup_mask.any():
    print(ret_only[dup_mask].sort_values(["customer_code", "visit_date", "product"]).to_string(index=False))

# ---- 2. Look for the matching SALE line: same customer + same product, quantity > 0, return 0 ----
sale_lines = lines[(lines["quantity"] > 0) & (lines["return_quantity"] == 0)]
pairs = ret_only.merge(sale_lines, on=["customer_code", "product"], how="left", suffixes=("_ret", "_sale"))
pairs["same_date"]     = pairs["visit_date_ret"] == pairs["visit_date_sale"]
pairs["same_number"]   = pairs["quantity_sale"] == pairs["return_quantity_ret"]
pairs["days_apart"]    = (pairs["visit_date_sale"] - pairs["visit_date_ret"]).dt.days
pairs["rows_apart"]    = pairs["row_no_sale"] - pairs["row_no_ret"]
pairs["adjacent_row"]  = pairs["rows_apart"].abs() == 1

# For each return-only row keep its BEST partner: same date first, then same number, then closest row
pairs = (pairs.sort_values(["row_no_ret", "same_date", "same_number", "rows_apart"],
                           ascending=[True, False, False, True],
                           key=lambda s: s.abs() if s.name == "rows_apart" else s)
              .drop_duplicates("row_no_ret"))

has_partner  = pairs["quantity_sale"].notna()
same_visit   = has_partner & pairs["same_date"]
exact_split  = same_visit & pairs["same_number"]
neighbour    = exact_split & pairs["adjacent_row"]

print("\n2. Matching sale line for the same customer and product:")
print("   any date                                   :", pct(has_partner.sum()))
print("   SAME date (same visit)                     :", pct(same_visit.sum()))
print("   same date AND sold qty == returned qty     :", pct(exact_split.sum()), " <- split record")
print("   ...and the sale is the row directly next to it:", pct(neighbour.sum()))
print("   matching sale only on a DIFFERENT date     :", pct((has_partner & ~pairs["same_date"]).sum()))
print("   no matching sale line at all               :", pct((~has_partner).sum()))

# ---- 3. Show each return-only row next to its partner, with both dates ----
show = pairs[["customer_code", "product", "visit_date_ret", "quantity_ret", "return_quantity_ret",
              "visit_date_sale", "quantity_sale", "return_quantity_sale",
              "same_date", "same_number", "days_apart", "rows_apart"]]
show.columns = ["customer", "product", "date (return row)", "qty", "return",
                "date (sale row)", "qty ", "return ", "same date?", "same number?", "days apart", "rows apart"]
print("\n3. Every return-only row with its best-matching sale line:")
print(show.sort_values(["same date?", "customer"], ascending=[False, True]).to_string(index=False))

print("\ndf unchanged:", df.shape)   # should still be (984292, 40)


In [ ]:
# ==========================================================================
# Do reps record the SALE and the RETURN of empties on separate lines?
# Read-only: works on a copy of df, nothing is changed or saved.
# ==========================================================================
cols = ["customer_code", "visit_date", "product", "quantity", "return_quantity"]
lines = df[cols].copy()
lines["line_no"] = df.index                     # position in the file, to test "the row beneath"

# The return-only rows: nothing bought on that line, empties handed back
ret_only  = lines[(lines["quantity"] == 0) & (lines["return_quantity"] > 0)]
sale_only = lines[(lines["quantity"] > 0) & (lines["return_quantity"] == 0)]
total = len(ret_only)

# --- 1. Duplicates among the return-only rows -----------------------------
n_dupes = ret_only.duplicated(subset=cols, keep=False).sum()

# --- 2. For each return-only row, find its closest matching sale line -----
#     (same customer + same product; prefer same date, then same number, then nearest line)
pairs = ret_only.merge(sale_only, on=["customer_code", "product"], how="left", suffixes=("_ret", "_sale"))
pairs["same_date"]   = pairs["visit_date_sale"] == pairs["visit_date_ret"]
pairs["same_number"] = pairs["quantity_sale"] == pairs["return_quantity_ret"]
pairs["days"]        = (pairs["visit_date_sale"] - pairs["visit_date_ret"]).dt.days
pairs["lines_apart"] = (pairs["line_no_sale"] - pairs["line_no_ret"]).abs()
pairs = (pairs.sort_values(["line_no_ret", "same_date", "same_number", "lines_apart"],
                           ascending=[True, False, False, True])
              .drop_duplicates("line_no_ret"))

# --- 3. Give every row a plain-language verdict ---------------------------
def verdict(r):
    if pd.isna(r["quantity_sale"]):
        return "No sale line found"
    if r["same_date"] and r["same_number"]:
        return "Split record (same visit)"
    if r["same_date"]:
        return "Same visit, numbers differ"
    if -30 <= r["days"] < 0:
        return "Empties from an earlier purchase"
    if 0 < r["days"] <= 30:
        return "Sale came after the return"
    return "No close match (>30 days)"

pairs["Verdict"] = pairs.apply(verdict, axis=1)

verdict_order = ["Split record (same visit)", "Empties from an earlier purchase",
                 "Same visit, numbers differ", "Sale came after the return",
                 "No close match (>30 days)", "No sale line found"]
meaning = {
    "Split record (same visit)":        "Sale and return typed on two lines of the SAME visit, same number",
    "Empties from an earlier purchase": "Customer bought earlier, brought the empties back on a later visit",
    "Same visit, numbers differ":       "Same visit, but sold and returned numbers are not equal",
    "Sale came after the return":       "Empties returned first, the matching purchase came later",
    "No close match (>30 days)":        "Nearest purchase of that product is over a month away - unrelated",
    "No sale line found":               "This customer never bought this product in the data",
}

# --- 4. Summary table -----------------------------------------------------
summary = (pairs["Verdict"].value_counts()
                .reindex(verdict_order, fill_value=0)
                .rename("Rows").to_frame())
summary["% of return-only rows"] = (summary["Rows"] / total * 100).round(1).astype(str) + "%"
summary["What it means"] = summary.index.map(meaning)
summary = summary[summary["Rows"] > 0]

adjacent = ((pairs["Verdict"] == "Split record (same visit)") & (pairs["lines_apart"] == 1)).sum()

print(f"Return-only rows checked: {total}")
print(f"Duplicates among them:    {n_dupes} ({n_dupes / total:.1%})")
print(f"Split records where the sale is the line DIRECTLY beneath/above: {adjacent} ({adjacent / total:.1%})")
display(summary)

# --- 5. Detail table: one row per return-only line, in plain words --------
pairs["_order"] = pd.Categorical(pairs["Verdict"], categories=verdict_order, ordered=True)
pairs = pairs.sort_values(["_order", "customer_code", "visit_date_ret"])   # sort on real dates, before formatting

fmt = lambda d: d.strftime("%d %b %Y") if pd.notna(d) else "-"
detail = pd.DataFrame({
    "Verdict":               pairs["Verdict"],
    "Customer":              pairs["customer_code"],
    "Product":               pairs["product"],
    "Returned on":           pairs["visit_date_ret"].map(fmt),
    "Empties returned":      pairs["return_quantity_ret"].astype(int),
    "Matching sale on":      pairs["visit_date_sale"].map(fmt),
    "Quantity sold":         pairs["quantity_sale"],
    "Same visit?":           pairs["same_date"].map({True: "Yes", False: "No"}),
    "Same number?":          pairs["same_number"].map({True: "Yes", False: "No"}),
    "Days from sale to return": (-pairs["days"]).astype("Int64"),
    "Lines apart in file":   pairs["lines_apart"].astype("Int64"),
})
detail = detail.reset_index(drop=True)
detail.index += 1

print("\nEvery return-only row and its matching sale line "
      "(Days from sale to return: 0 = same visit, 3 = bought 3 days before returning):")
display(detail)

print("df unchanged:", df.shape)   # should still be (984292, 40)


In [ ]:
# ==========================================================================
# Which categories are the 36 split records in? (read-only; uses `pairs` from Cell 1 and df)
# ==========================================================================
split = pairs[pairs["Verdict"] == "Split record (same visit)"]
info_cols = ["Category", "Category2", "Sub_Category", "Brand", "Product_Code"]

ret_side  = df.loc[split["line_no_ret"],  info_cols].reset_index(drop=True).add_suffix(" (return line)")
sale_side = df.loc[split["line_no_sale"].astype(int), info_cols].reset_index(drop=True).add_suffix(" (sale line)")
both = pd.concat([split[["customer_code", "visit_date_ret", "product"]].reset_index(drop=True),
                  ret_side, sale_side], axis=1)

n = len(both)
same_cat = (both["Category (return line)"] == both["Category (sale line)"]).sum()
print(f"Split records: {n}")
print(f"Both halves in the SAME category: {same_cat} of {n} ({same_cat / n:.1%})")

by_cat = (both["Category (return line)"].value_counts()
              .rename("Rows").to_frame())
by_cat["% of split records"] = (by_cat["Rows"] / n * 100).round(1).astype(str) + "%"
print("\nCategory of the split records:")
display(by_cat)

print("\nBreakdown by Category2 / Sub_Category / Brand:")
display(both.groupby(["Category2 (return line)", "Sub_Category (return line)", "Brand (return line)"])
            .size().rename("Rows").reset_index())

print("\nProducts involved:")
display(both["product"].value_counts().rename("Rows").to_frame())


In [ ]:
# ==========================================================================
# Diagram: what the return-only rows really are (uses `pairs`, `summary`, `total` from the cell above)
# ==========================================================================

colors = {                                   # fixed colour per verdict (colour-blind-safe order)
    "Split record (same visit)":        "#2a78d6",   # blue
    "Empties from an earlier purchase": "#eb6834",   # orange
    "Same visit, numbers differ":       "#1baf7a",   # aqua
    "Sale came after the return":       "#eda100",   # yellow
    "No close match (>30 days)":        "#a3a29d",   # grey = no real partner
    "No sale line found":               "#cfcec8",
}
ink, ink_soft = "#0b0b0b", "#52514e"

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5), gridspec_kw={"width_ratios": [1, 1.25]})

# ---- Left: how many rows fall in each verdict -----------------------------
s = summary.iloc[::-1]                                   # biggest group on top
bars = ax1.barh(s.index, s["Rows"], color=[colors[v] for v in s.index], height=0.6)
for bar, (n, p) in zip(bars, zip(s["Rows"], s["% of return-only rows"])):
    ax1.text(bar.get_width() + 0.6, bar.get_y() + bar.get_height() / 2,
             f"{n}  ({p})", va="center", fontsize=11, color=ink)
ax1.set_xlim(0, s["Rows"].max() * 1.35)
ax1.set_title(f"What are the {total} return-only rows?", loc="left", fontsize=13, color=ink)
ax1.set_xlabel("Number of rows", color=ink_soft)
ax1.tick_params(colors=ink_soft)
ax1.grid(axis="x", color="#e6e5e0", linewidth=0.8); ax1.set_axisbelow(True)
for side in ["top", "right", "left"]:
    ax1.spines[side].set_visible(False)

# ---- Right: when was the matching sale, relative to the return? ----------
plot = pairs[pairs["days"].between(-30, 30)].copy()
plot["days_before"] = -plot["days"]                      # +3 = sale 3 days BEFORE the return
# Stack the dots in columns (one dot per row) so every row can be counted - no overlap
plot["_order"] = pd.Categorical(plot["Verdict"], categories=list(colors), ordered=True)
plot = plot.sort_values(["days_before", "_order"])
plot["level"] = plot.groupby("days_before").cumcount() + 1
for v in [v for v in colors if v in plot["Verdict"].unique()]:
    g = plot[plot["Verdict"] == v]
    ax2.scatter(g["days_before"], g["level"], s=60, color=colors[v], edgecolor="white", linewidth=1.5,
                label=f"{v} ({len(g)})", zorder=3)
ax2.axvline(0, color=ink_soft, linewidth=1, linestyle="--", zorder=1)
ax2.text(0.4, plot["level"].max() + 0.5, "same visit", color=ink_soft, fontsize=10)
ax2.set_xlim(-16, 16); ax2.set_ylim(0, plot["level"].max() + 2.5)
ax2.set_ylabel("Number of rows", color=ink_soft)
ax2.set_xlabel("Days from the matching sale to the return   (right = sale happened BEFORE the return)",
               color=ink_soft)
ax2.set_title("Each dot = one return-only row", loc="left", fontsize=13, color=ink)
ax2.tick_params(colors=ink_soft)
ax2.grid(axis="y", color="#e6e5e0", linewidth=0.8); ax2.set_axisbelow(True)
for side in ["top", "right"]:
    ax2.spines[side].set_visible(False)
ax2.legend(loc="upper left", bbox_to_anchor=(0, -0.18), ncol=2, frameon=False, fontsize=10)
far = (~pairs["days"].between(-30, 30) & pairs["days"].notna()).sum()
if far:
    ax2.text(1, -0.16, f"{far} row(s) with a match >30 days away not shown", transform=ax2.transAxes,
             ha="right", fontsize=9, color=ink_soft)

plt.tight_layout()
plt.show()


In [ ]:
# ==========================================================================
# Show the WHOLE detail table: all 68 rows, all columns, coloured by verdict
# (display only - uses `detail` from the cell above, changes nothing)
# ==========================================================================
row_colors = {                               # light tints of the diagram colours
    "Split record (same visit)":        "#dbe8f8",   # blue
    "Empties from an earlier purchase": "#fbe1d6",   # orange
    "Same visit, numbers differ":       "#d4f0e5",   # aqua
    "Sale came after the return":       "#fbecc7",   # yellow
    "No close match (>30 days)":        "#ecebe7",   # grey
    "No sale line found":               "#f5f5f2",
}

def colour_row(row):
    return [f"background-color: {row_colors.get(row['Verdict'], 'white')}; color: #0b0b0b"] * len(row)

styled = (detail.style
          .apply(colour_row, axis=1)
          .format({"Quantity sold": "{:,.0f}"}, na_rep="-")
          .set_properties(**{"text-align": "left", "white-space": "nowrap", "font-size": "12px"})
          .set_table_styles([
              {"selector": "th", "props": [("background-color", "#52514e"), ("color", "white"),
                                           ("text-align", "left"), ("position", "sticky"), ("top", "0")]},
          ])
          .set_caption(f"All {len(detail)} return-only rows and their matching sale line"))

with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.width", None):
    display(summary)
    display(styled)


In [ ]:
# ==========================================================================
# STEP 1 - check 5: lines where MORE was returned than was bought
# Read-only: looks at df, changes and saves nothing.
# ==========================================================================
cols = ["customer_code", "salesperson", "visit_date", "product", "Product_Code", "Category",
        "quantity", "return_quantity"]
over = df.loc[(df["quantity"] > 0) & (df["return_quantity"] > df["quantity"]), cols]
print(f"Lines where returns are larger than quantity sold: {len(over)}")
print("Categories:", over["Category"].value_counts().to_dict())

rows = []
for idx, r in over.iterrows():
    same_cp  = df[(df["customer_code"] == r["customer_code"]) & (df["product"] == r["product"])]
    same_day = same_cp[same_cp["visit_date"] == r["visit_date"]]
    day_bought, day_returned = same_day["quantity"].sum(), same_day["return_quantity"].sum()
    if len(same_day) > 1 and day_bought >= day_returned:
        verdict = "Split record - other line that day covers it"
    elif same_cp["quantity"].sum() >= same_cp["return_quantity"].sum():
        verdict = "Empties from earlier purchases - normal"
    else:
        verdict = "Customer returns more than they ever bought - ask EABL"
    rows.append({
        "Row": idx,
        "Customer": r["customer_code"],
        "Product": r["product"],
        "Category": r["Category"],
        "Date": r["visit_date"].strftime("%d %b %Y"),
        "Bought (this line)": r["quantity"],
        "Returned (this line)": r["return_quantity"],
        "Lines that day": len(same_day),
        "Bought that day (all lines)": day_bought,
        "Returned that day (all lines)": day_returned,
        "Bought in whole data": same_cp["quantity"].sum(),
        "Returned in whole data": same_cp["return_quantity"].sum(),
        "Verdict": verdict,
    })

result = pd.DataFrame(rows).set_index("Row")
with pd.option_context("display.max_columns", None, "display.width", 250):
    display(result)

# Every line that customer had for that product on that day
for idx, r in over.iterrows():
    same_day = df[(df["customer_code"] == r["customer_code"]) & (df["product"] == r["product"])
                  & (df["visit_date"] == r["visit_date"])]
    print(f"\nRow {idx} - {r['customer_code']}, {r['product']}, {r['visit_date']:%d %b %Y}:")
    display(same_day[["quantity", "return_quantity"]])

print("df unchanged:", df.shape)


In [ ]:
# Columns we agreed are not needed for forecasting
columns_to_drop = [
    "source_file",
    "salesperson",
    "cust_type",
    "Product_Description",
    "Handheld_Description",
    "Default_UOM",
    "Pack_Format",
    "Bottles_Per_Case",
    "Pack_Size_mls",
    'FY25_July_NSV_Conversion_1st', 
    'FY25_November_NSV_Conversion_1st', 
    'FY25_January_NSV_Conversion_1st', 
    'FY26_October_NSV_Conversion_18th', 
    'FY25_EX_KBL_Price_Per_Case', 
    'FY26_October_NSV_Conversion_18th_Updated', 
    'FY26_June_NSV_Conversion_15th', 'FY26_EX_KBL_Price_Per_Case'
]

# drop() with a new variable name leaves 'df' (and the original CSV) untouched
df_trimmed = df.drop(columns=columns_to_drop)
# Start training data from April 1, 2025 — the confirmed clean point.
# March 1-5: real but incomplete (partial weeks, misleadingly low).
# March 6-31: no raw source file exists for this period at all (confirmed).
# April 1 onward: verified clean — no empty weeks, no unexpected missing days,
# no gaps in source-file coverage anywhere through the end of the dataset.
rows_before = len(df_trimmed)
df_trimmed = df_trimmed[df_trimmed['visit_date'] >= pd.Timestamp('2025-04-01')]
rows_after = len(df_trimmed)
print(f"Dropped {rows_before - rows_after} rows from the unreliable opening period")
print(f"New date range starts: {df_trimmed['visit_date'].min()}")

# Save to a NEW file — original sales_products_merged_cleaned.csv is never overwritten
df_trimmed.to_csv("../eabl parsed csv/sales_products_forecast_ready.csv", index=False)

print("Original columns:", len(df.columns))
print("Trimmed columns:", len(df_trimmed.columns))
print(list(df_trimmed.columns))


In [ ]:
# ==========================================================================
# STEP 3 - Drop the incomplete final week (FPP2 3.2, calendar adjustments)
# The data ends on Tuesday 30 June 2026, so the week of 29 June holds only
# 2 trading days (Mon 29, Tue 30) instead of 6. Summed into a week, it looks
# like a sales collapse that never happened, and it would mislead both the
# cross-validation and the final forecast. We end on Sunday 28 June 2026,
# the last COMPLETE week (Mon 22 - Sun 28 June).
# Uses df_trimmed from the cell above; df and the original CSV are untouched.
# ==========================================================================
last_full_day = pd.Timestamp("2026-07-12")    # Sunday = end of the last complete Mon-Sun week (data ends Sat 11 Jul; outlets close Sundays)

dropped = df_trimmed[df_trimmed["visit_date"] > last_full_day]
print(f"Rows in the partial week being dropped: {len(dropped):,}")
print(dropped["visit_date"].dt.date.value_counts().sort_index().to_string())
print(f"Cases in those 2 days: {dropped['quantity_cases'].sum():,.0f}")

df_trimmed = df_trimmed[df_trimmed["visit_date"] <= last_full_day]

# Check: the last week must now be a full week
weeks = df_trimmed.groupby(df_trimmed["visit_date"].dt.to_period("W").dt.start_time)["quantity_cases"].sum()
print(f"\nNew date range: {df_trimmed['visit_date'].min().date()} to {df_trimmed['visit_date'].max().date()}")
print("Last 4 weeks (cases):")
print(weeks.tail(4).round(0).to_string())

# Re-save the forecast-ready file with the end cut applied
df_trimmed.to_csv("../eabl parsed csv/sales_products_forecast_ready.csv", index=False)
print("\nSaved:", df_trimmed.shape)


In [ ]:
# ==========================================================================
# STEP 2 (C1) - Part 1: FULL INSPECTION of the paired codes + DECISION
# Read-only: nothing in df_trimmed, df or any file is changed or saved.
#
# Where each input comes from:
#   df_trimmed  -> cell 48 ("columns_to_drop") + Step 3 cell (cell 49): the forecast-ready
#                  daily rows, exactly what is saved as sales_products_forecast_ready.csv
#                  and read by the Prep notebook.
#   pair_master -> read here READ-ONLY from ../eabl parsed csv/product_master_cleaned.csv
#                  (EABL's product master: pack size, bottles per case, price per case).
#                  Named pair_master so it does NOT overwrite `pm` (used in cells 21 and 31).
#   pd, np      -> pandas / numpy, imported at the top of the notebook.
#   display     -> Jupyter's built-in table display.
# Every new name in this cell starts with pair_ / PAIR_ so it cannot clash with other cells.
#
# B-code    = Product_Code starting with "B" (sold per single bottle), e.g. B681232
# Case code = the same number without the B (sold per case of 25), e.g. 681232
#
# DECISION RULES
#   Identity tests - ALL must PASS for a pair, otherwise that pair must NOT be merged:
#     T1 Same unit      : every B-code quantity is a whole number of bottles written in cases
#                         (multiple of 1/25 = 0.04) and both codes use Cases_Conversion = 1
#     T2 Same drink     : same pack size (ml) and same bottles per case in the product master
#     T3 Same price     : same ex-factory price per case, FY25 and FY26 (within 1%)
#     T4 Same hierarchy : same Category2, Sub_Category and Brand
#   Benefit tests - evidence that merging helps forecasting (reported, not required):
#     B1 Steadier       : merged weekly series has a lower CV than the case series alone
#                         (CV = standard deviation / mean of weekly cases; lower = steadier)
#     B2 Fewer zeros    : merged series has no more zero-sale weeks than the case series
#     B3 Same customers : % of the drink's customers who buy BOTH forms
# ==========================================================================

# ---- Safety checks: stop if df_trimmed is not the expected table ----
assert {"Product_Code", "product", "quantity", "quantity_cases", "Cases_Conversion", "visit_date",
        "customer_code", "Category2", "Sub_Category", "Brand"}.issubset(df_trimmed.columns), \
    "`df_trimmed` is not the forecast-ready table - re-run cell 48 and the Step 3 cell"
assert "Original_Product_Code" not in df_trimmed.columns, \
    "paired codes are already merged - re-run cell 48 and the Step 3 cell to inspect the raw codes"

PAIR_PRICE_COLS = ["FY25_EX_KBL_Price_Per_Case", "FY26_EX_KBL_Price_Per_Case"]
pair_master = (pd.read_csv("../eabl parsed csv/product_master_cleaned.csv", dtype={"Product_Code": str})
                 .drop_duplicates("Product_Code").set_index("Product_Code"))

pair_codes     = df_trimmed["Product_Code"].astype(str)
pair_is_bottle = pair_codes.str.startswith("B")
pair_base      = pair_codes.str.lstrip("B")
pair_rows = df_trimmed[pair_base.isin(set(pair_base[pair_is_bottle]))].assign(
    base_code=pair_base,
    form=pair_is_bottle.map({True: "bottle", False: "case"}),
    week=pd.to_datetime(df_trimmed["visit_date"]).dt.to_period("W").dt.start_time)
pair_weeks = pd.date_range(pair_rows["week"].min(), pair_rows["week"].max(), freq="W-MON")


def pair_pf(ok):
    return "PASS" if ok else "FAIL"


def pair_cv(series):
    return series.std() / series.mean() if series.mean() > 0 else np.nan


def pair_show(title, table_rows, note):
    print("\n" + "=" * 90 + f"\n{title}\n" + "=" * 90 + f"\n{note}")
    with pd.option_context("display.max_rows", None, "display.max_columns", None,
                           "display.width", 300, "display.max_colwidth", 50):
        display(pd.DataFrame(table_rows).rename(index=lambda n: n + 1))


# ---- 0. Overview ----
print("=" * 90 + "\n0. OVERVIEW\n" + "=" * 90)
print(f"B-codes in the product master : {pair_master.index.str.startswith('B').sum()}")
print(f"B-codes sold in the data      : {pair_codes[pair_is_bottle].nunique()} "
      f"({pair_is_bottle.sum():,} rows, {df_trimmed.loc[pair_is_bottle, 'quantity_cases'].sum():,.1f} cases)")
print(f"Products affected             : {pair_rows['base_code'].nunique()} "
      f"| product codes today: {pair_codes.nunique()}")

pair_t1, pair_t2, pair_t3, pair_t4, pair_ben = [], [], [], [], []
for pair_case_code, pair_g in pair_rows.groupby("base_code"):
    pair_bcode    = "B" + pair_case_code
    pair_gb       = pair_g[pair_g["form"] == "bottle"]
    pair_gc       = pair_g[pair_g["form"] == "case"]
    pair_has_case = len(pair_gc) > 0
    pair_mb = pair_master.loc[pair_bcode] if pair_bcode in pair_master.index else None
    pair_mc = pair_master.loc[pair_case_code] if pair_case_code in pair_master.index else None
    pair_in_master = pair_mb is not None and pair_mc is not None

    # T1 - same unit
        # bottles per case for THIS pair, read from EABL's product master - no assumed value.
    # If the B-code is missing from the master, the case size is unknown and T1 FAILS.
    pair_per_case = pair_mb["Bottles_Per_Case"] if pair_mb is not None else np.nan
    pair_whole = ((pair_gb["quantity"] * pair_per_case).round(6) % 1 == 0).mean() if pd.notna(pair_per_case) else 0.0
    pair_conv_ok = set(pair_g["Cases_Conversion"].round(6)) == {1.0}
    pair_t1.append({
        "Case code": pair_case_code, "Bottle rows": len(pair_gb),
        "% whole bottles": round(pair_whole * 100, 1),
        "Smallest bottle qty": pair_gb["quantity"].min(),
        "Typical bottle qty": pair_gb["quantity"].median(),
        "Typical case qty": pair_gc["quantity"].median() if pair_has_case else None,
        "Cases_Conversion used": sorted(set(pair_g["Cases_Conversion"].round(4))),
        "T1": pair_pf(pair_whole == 1 and pair_conv_ok)})

    # T2 - same drink
    pair_t2_ok = pair_in_master and pair_mb["Pack_Size_mls"] == pair_mc["Pack_Size_mls"] \
                 and pair_mb["Bottles_Per_Case"] == pair_mc["Bottles_Per_Case"]
    pair_t2.append({
        "Case code": pair_case_code,
        "Case product": pair_gc["product"].iloc[0] if pair_has_case
                        else (pair_mc["Product_Description"] if pair_mc is not None else "-"),
        "Bottle product": pair_gb["product"].iloc[0],
        "ml (case / bottle)": f"{pair_mc['Pack_Size_mls'] if pair_mc is not None else '?'} / "
                              f"{pair_mb['Pack_Size_mls'] if pair_mb is not None else '?'}",
        "Per case (case / bottle)": f"{pair_mc['Bottles_Per_Case'] if pair_mc is not None else '?'} / "
                                    f"{pair_mb['Bottles_Per_Case'] if pair_mb is not None else '?'}",
        "UOM (case / bottle)": f"{pair_mc['Default_UOM'] if pair_mc is not None else '?'} / "
                               f"{pair_mb['Default_UOM'] if pair_mb is not None else '?'}",
        "T2": pair_pf(pair_t2_ok)})

    # T3 - same price
    pair_t3_ok = pair_in_master and all(np.isclose(pair_mb[col], pair_mc[col], rtol=0.01) for col in PAIR_PRICE_COLS)
    pair_t3.append({
        "Case code": pair_case_code,
        "FY25 price (case code)":   round(pair_mc[PAIR_PRICE_COLS[0]], 2) if pair_mc is not None else None,
        "FY25 price (bottle code)": round(pair_mb[PAIR_PRICE_COLS[0]], 2) if pair_mb is not None else None,
        "FY26 price (case code)":   round(pair_mc[PAIR_PRICE_COLS[1]], 2) if pair_mc is not None else None,
        "FY26 price (bottle code)": round(pair_mb[PAIR_PRICE_COLS[1]], 2) if pair_mb is not None else None,
        "T3": pair_pf(pair_t3_ok)})

    # T4 - same hierarchy
    pair_t4_ok = bool((pair_g[["Category2", "Sub_Category", "Brand"]].nunique() <= 1).all())
    pair_t4.append({
        "Case code": pair_case_code,
        **{col: " | ".join(sorted(pair_g[col].astype(str).unique())) for col in ["Category2", "Sub_Category", "Brand"]},
        "T4": pair_pf(pair_t4_ok)})

    # B1-B3 - benefit of merging (weekly series, Mon-Sun weeks, missing weeks = 0)
    pair_wk = (pair_g.groupby(["week", "form"])["quantity_cases"].sum().unstack(fill_value=0)
                     .reindex(pair_weeks, fill_value=0).reindex(columns=["bottle", "case"], fill_value=0))
    pair_merged_wk = pair_wk["bottle"] + pair_wk["case"]
    pair_cust = pair_g.groupby("customer_code")["form"].nunique()
    pair_ben.append({
        "Case code": pair_case_code, "Brand": pair_g["Brand"].iloc[0],
        "Avg cases/wk (case)": round(pair_wk["case"].mean(), 2),
        "Avg cases/wk (bottle)": round(pair_wk["bottle"].mean(), 2),
        "Bottle share %": round(pair_wk["bottle"].sum() / pair_merged_wk.sum() * 100, 1),
        "CV case": round(pair_cv(pair_wk["case"]), 3) if pair_has_case else None,
        "CV bottle": round(pair_cv(pair_wk["bottle"]), 3),
        "CV merged": round(pair_cv(pair_merged_wk), 3),
        "B1 steadier": pair_pf(pair_cv(pair_merged_wk) < pair_cv(pair_wk["case"])) if pair_has_case else "n/a",
        "Zero wks (case)": int((pair_wk["case"] == 0).sum()) if pair_has_case else None,
        "Zero wks (merged)": int((pair_merged_wk == 0).sum()),
        "B2 fewer zeros": pair_pf((pair_merged_wk == 0).sum() <= (pair_wk["case"] == 0).sum()) if pair_has_case else "n/a",
        "Customers": len(pair_cust),
        "B3 buy both %": round((pair_cust == 2).mean() * 100, 1),
        "Has case sales": pair_has_case})

pair_show("T1  SAME UNIT", pair_t1,
          "PASS = 100% whole bottles (qty x 25 is a whole number) and Cases_Conversion = 1 for both codes")
pair_show("T2  SAME DRINK", pair_t2,
          "PASS = same ml and same bottles per case in the product master (UOM: CAS = case code, BTL = bottle code)")
pair_show("T3  SAME PRICE", pair_t3,
          "PASS = same ex-factory price per case in FY25 and FY26 (within 1%)")
pair_show("T4  SAME HIERARCHY", pair_t4,
          "PASS = one Category2, one Sub_Category and one Brand across both codes")
pair_show("B1-B3  BENEFIT OF MERGING", pair_ben,
          f"Weekly series over {len(pair_weeks)} weeks. Lower CV = steadier. Fewer zero weeks = less intermittent.")

# ---- Decision per pair ----
pair_decision = pd.DataFrame({
    "Case code": [row["Case code"] for row in pair_t1],
    "Brand":     [row["Brand"] for row in pair_ben],
    "T1": [row["T1"] for row in pair_t1], "T2": [row["T2"] for row in pair_t2],
    "T3": [row["T3"] for row in pair_t3], "T4": [row["T4"] for row in pair_t4],
    "B1": [row["B1 steadier"] for row in pair_ben], "B2": [row["B2 fewer zeros"] for row in pair_ben]})
pair_ident_ok = (pair_decision[["T1", "T2", "T3", "T4"]] == "PASS").all(axis=1)
pair_case_ok  = pd.Series([row["Has case sales"] for row in pair_ben])
pair_gain     = (pair_decision[["B1", "B2"]] == "PASS").any(axis=1)
pair_decision["Verdict"] = np.select(
    [~pair_ident_ok, pair_ident_ok & ~pair_case_ok, pair_ident_ok & pair_gain],
    ["DO NOT MERGE - identity test failed, check with EABL",
     "RENAME ONLY - no case sales; becomes the case code",
     "MERGE - same product, merged series steadier"],
    default="MERGE - same product (no forecasting gain, no harm)")
pair_show("DECISION PER PAIR", pair_decision,
          "Identity tests T1-T4 must all PASS; B1-B2 show how much merging helps")

# ---- Overall ----
print("\n" + "=" * 90 + "\nOVERALL\n" + "=" * 90)
print("Identity tests (T1-T4) passed:", int(pair_ident_ok.sum()), "of", len(pair_decision))
print(pair_decision["Verdict"].value_counts().to_string())
pair_go = pair_decision["Verdict"].str.startswith(("MERGE", "RENAME")).all()
print(f"\nIf merged: product codes {pair_codes.nunique()} -> {pair_base.nunique()}; "
      f"rows and total cases stay the same ({len(df_trimmed):,} rows, {df_trimmed['quantity_cases'].sum():,.2f} cases)")
print("RESULT:", "MERGE IS SUPPORTED BY THE EVIDENCE - you can ask for Part 2 (merge + save)" if pair_go
      else "DO NOT MERGE YET - fix or exclude the pairs marked DO NOT MERGE")
print("Still to confirm with EABL (business, not data): do you plan bottles and cases as one product?")
print("df_trimmed unchanged:", df_trimmed.shape)


In [ ]:
# ==========================================================================
# STEP 2 (C1) - Part 2: MERGE each B-code (per bottle) into its case code, then SAVE
#
# Why: the inspection cell above proved (T1-T4 = 21/21 PASS) that each B-code is the SAME
# drink as its case code - same unit, same ml and case size, same price, same brand.
#
# Where each input comes from:
#   df_trimmed -> cell 48 + Step 3 cell (cell 49): the forecast-ready daily rows.
#   pair_go    -> the inspection cell above (True only if every pair is MERGE or RENAME).
#   pd         -> pandas, imported at the top of the notebook.
#
# What it changes in df_trimmed:
#   - Product_Code of B-code rows becomes the case code (B681232 -> 681232)
#   - NEW column Original_Product_Code = the code as recorded (nothing is lost)
#   - NEW column Pack_Form_Sold        = "bottle" or "case"
#   No rows added or removed. Quantities NOT changed.
# What it writes: ../eabl parsed csv/sales_products_forecast_ready.csv (the file Step 3 saves).
# NOT touched: df, sales_products_merged_cleaned.csv, the flags file, known_spikes.csv.
# ==========================================================================

# ---- Safety checks ----
assert "pair_go" in globals() and pair_go, \
    "run the inspection cell above first - it must say 'MERGE IS SUPPORTED BY THE EVIDENCE'"
assert "Original_Product_Code" not in df_trimmed.columns, \
    "already merged - re-run cell 48 and the Step 3 cell first, then this cell ONCE"

merge_codes  = df_trimmed["Product_Code"].astype(str)        # codes as recorded
merge_bottle = merge_codes.str.startswith("B")               # True = per-bottle row
merge_base   = merge_codes.str.lstrip("B")                   # case code (B681232 -> 681232)

merge_rows_before  = len(df_trimmed)
merge_cases_before = df_trimmed["quantity_cases"].sum()
merge_codes_before = merge_codes.nunique()

# ---- The merge ----
df_trimmed["Original_Product_Code"] = merge_codes
df_trimmed["Pack_Form_Sold"]        = merge_bottle.map({True: "bottle", False: "case"})
df_trimmed["Product_Code"]          = merge_base

# ---- Checks: nothing lost, nothing added ----
print(f"Rows:          {merge_rows_before:,} -> {len(df_trimmed):,}   (must be equal)")
print(f"Total cases:   {merge_cases_before:,.2f} -> {df_trimmed['quantity_cases'].sum():,.2f}   (must be equal)")
print(f"Product codes: {merge_codes_before} -> {df_trimmed['Product_Code'].nunique()}   "
      f"({merge_codes[merge_bottle].nunique()} B-codes merged or renamed)")
print("B-codes left:", int(df_trimmed["Product_Code"].astype(str).str.startswith("B").sum()), "(must be 0)")
assert len(df_trimmed) == merge_rows_before, "row count changed - stop"
assert abs(df_trimmed["quantity_cases"].sum() - merge_cases_before) < 1e-6, "total cases changed - stop"

# ---- Save for the Prep notebook ----
df_trimmed.to_csv("../eabl parsed csv/sales_products_forecast_ready.csv", index=False)
print("\nSaved sales_products_forecast_ready.csv:", df_trimmed.shape)

print("\nExample - Balozi rows after the merge (both forms now share code 681232):")
display(df_trimmed[df_trimmed["Product_Code"] == "681232"]
          .groupby(["Product_Code", "Original_Product_Code", "Pack_Form_Sold"])["quantity_cases"]
          .agg(rows="size", cases="sum").round(2))

In [ ]:
# ==========================================================================
# STEP 3 (D1) - Drop the incomplete FIRST week (Mon 31 Mar - Sun 6 Apr 2025)
#
# WHY (FPP2 3.2 calendar adjustments, p.61-62; 12.9 missing values, p.486):
#   Weeks must be comparable (6 trading days, Mon-Sat). The data starts on Tue 1 Apr 2025,
#   so Mon 31 Mar is missing, and Thu 3 Apr is almost empty (143 lines vs ~1,825 on a
#   normal Thursday). The week is 0.72 x a normal week - the LOWEST of all 67 weeks.
#   This is missing recording, not lower demand. The data will start on Mon 7 Apr 2025.
#
# WHERE EACH INPUT COMES FROM:
#   df_trimmed -> built in cell 50 (columns_to_drop + 1 Apr start cut), cut at the end in
#                 cell 51 (STEP 3, last_full_day = 12 Jul 2026), B-codes merged in cell 53 (C1).
#                 One row per sales line; columns used here:
#                   visit_date     - the sale date (datetime, set in cell 29)
#                   quantity_cases - quantity in cases (cell 2: quantity x Cases_Conversion)
#                   Product_Code   - the product (after the C1 merge, 203 codes)
#   pd         -> pandas, imported in cell 1.
#   display    -> Jupyter's built-in table display (no import needed in a notebook).
#
# WHAT IT CHANGES: removes only the rows dated 1-6 Apr 2025 from df_trimmed (10,203 rows).
# WHAT IT WRITES:  ../eabl parsed csv/sales_products_forecast_ready.csv
#                  (the same file cells 50, 51 and 53 write; Prep cell 1 reads it).
# NOT TOUCHED:     df, sales_products_merged_cleaned.csv, the flags file, known_spikes.csv.
# All new names start with d1_ / D1_ so nothing clashes with other cells.
# ==========================================================================
D1_FIRST_FULL_WEEK = pd.Timestamp("2025-04-07")      # Monday of the first complete Mon-Sun week

# ---- Safety checks: stop (and save nothing) if df_trimmed is not the table we expect ----
assert {"visit_date", "quantity_cases", "Product_Code", "Original_Product_Code"}.issubset(df_trimmed.columns), \
    "`df_trimmed` is not the forecast-ready table after C1 - re-run cells 50, 51, 52 and 53 first"
assert df_trimmed["visit_date"].max() <= pd.Timestamp("2026-07-12"), \
    "data runs past 12 Jul 2026 - run the STEP 3 cell (cell 51, end cut) first"
assert df_trimmed["Product_Code"].astype(str).str.startswith("B").sum() == 0, \
    "B-codes still present - run the C1 Part 2 cell (cell 53) first"

# ---- What will be dropped ----
d1_dates     = pd.to_datetime(df_trimmed["visit_date"])          # sale dates of df_trimmed
d1_drop      = d1_dates < D1_FIRST_FULL_WEEK                      # True for rows dated 1-6 Apr 2025
d1_rows_bef  = len(df_trimmed)                                    # rows before (for the check below)
d1_cases_bef = df_trimmed["quantity_cases"].sum()                 # cases before
d1_prod_bef  = df_trimmed["Product_Code"].nunique()               # products before (expect 203)

print(f"Rows being dropped (1-6 Apr 2025): {d1_drop.sum():,}  "
      f"({df_trimmed.loc[d1_drop, 'quantity_cases'].sum():,.1f} cases)")
if d1_drop.any():
    d1_by_day = (df_trimmed[d1_drop]
                 .groupby(d1_dates[d1_drop].dt.date)["quantity_cases"]
                 .agg(rows="size", cases="sum"))                  # one line per dropped day
    d1_by_day.index = pd.to_datetime(d1_by_day.index).strftime("%a %d %b %Y")
    display(d1_by_day.round(1))

# ---- Drop the rows ----
df_trimmed = df_trimmed[~d1_drop]

# ---- Checks after the drop ----
d1_weeks = (df_trimmed
            .groupby(pd.to_datetime(df_trimmed["visit_date"]).dt.to_period("W").dt.start_time)
            ["quantity_cases"].sum())                             # cases per Mon-Sun week
print(f"\nNew date range: {pd.to_datetime(df_trimmed['visit_date']).min().date()} "
      f"to {pd.to_datetime(df_trimmed['visit_date']).max().date()}   ({len(d1_weeks)} weeks)")
print(f"Rows:     {d1_rows_bef:,} -> {len(df_trimmed):,}")
print(f"Cases:    {d1_cases_bef:,.2f} -> {df_trimmed['quantity_cases'].sum():,.2f}")
print(f"Products: {d1_prod_bef} -> {df_trimmed['Product_Code'].nunique()}   (must be equal - no product lost)")
assert df_trimmed["Product_Code"].nunique() == d1_prod_bef, "a product would disappear - stop and check"
assert d1_weeks.index.min() == D1_FIRST_FULL_WEEK, "the first week is not 7 Apr 2025 - stop and check"
print("First 3 weeks (cases):"); print(d1_weeks.head(3).round(0).to_string())

# ---- Save (same file as cells 50, 51 and 53; read by Prep cell 1) ----
df_trimmed.to_csv("../eabl parsed csv/sales_products_forecast_ready.csv", index=False)
print("\nSaved sales_products_forecast_ready.csv:", df_trimmed.shape)


In [ ]:
df_trimmed.shape

In [ ]:
df_trimmed.isna().sum()

In [ ]:
df_trimmed

In [ ]:
check = pd.read_csv("../eabl parsed csv/sales_products_forecast_ready.csv")
print(check['visit_date'].min())  # should print 2025-03-31 or later


In [ ]:
check = pd.read_csv("../eabl parsed csv/sales_products_forecast_ready.csv")
print(check['visit_date'].min())  # should print 2025-03-31 or later
check

In [ ]:
# ==========================================================================
# CHECK - did Y_product / S_product pick up the C1 merge? (read-only)
# Reads the files saved by this notebook (cell 24). Nothing is changed or saved.
# ==========================================================================
ycheck_Y = pd.read_csv("../eabl parsed csv/Y_product.csv")
ycheck_S = pd.read_csv("../eabl parsed csv/S_product.csv", index_col=0)
ycheck_bottom = ycheck_Y["unique_id"].str.count("/") == 3

print("Y_product rows:", len(ycheck_Y), "(expect 17790)")
print("Series:", ycheck_Y["unique_id"].nunique(), "(expect 299)")
print("Products:", ycheck_Y.loc[ycheck_bottom, "unique_id"].nunique(), "(expect 202)")
print("B-code products left:", ycheck_Y["unique_id"].str.contains(r"/B\d{5,6}$").sum(), "(expect 0)")
print("S_product shape:", ycheck_S.shape, "(expect (299, 203))")
print("Tusker Ndimu 782264, week of 25 Aug 2025:",
      ycheck_Y.loc[(ycheck_Y["unique_id"] == "Beer/Mainstream/Tusker Ndimu/782264")
                   & (ycheck_Y["ds"] == "2025-08-25"), "y"].round(2).tolist(), "(expect [0.84])")


In [ ]:
# ==========================================================================
# CHECK - did Y_product / S_product pick up C1 + July + D1 + R1 (Returnable split)? (read-only)
# Reads the files saved by the PREP notebook. Nothing is changed or saved.
#
# Where each input comes from:
#   pd               -> pandas, imported in cell 1 of this notebook.
#   Y_product.csv    -> saved by Prep (hierarchy save cell): drinks only, long format: unique_id, ds, y.
#   S_product.csv    -> saved by Prep: row number index + unique_id + 188 product columns.
#   Y_returnable.csv -> saved by Prep cell R1: the 15 Returnable items, same format as Y_product.
# All names start with ycheck_ (used only in this cell).
# Expected numbers = after July (to 11 Jul 2026), C1 (B-codes merged), D1 (start 7 Apr 2025)
# and R1 (Returnable separated: drinks = 3 categories, 8 sub-categories, 83 brands, 188 products).
# ==========================================================================
ycheck_Y = pd.read_csv("../eabl parsed csv/Y_product.csv")
ycheck_S = pd.read_csv("../eabl parsed csv/S_product.csv", index_col=0)
ycheck_R = pd.read_csv("../eabl parsed csv/Y_returnable.csv")
ycheck_bottom = ycheck_Y["unique_id"].str.count("/") == 3          # product level = 3 slashes

ycheck_rows   = len(ycheck_Y)
ycheck_series = ycheck_Y["unique_id"].nunique()
ycheck_prods  = ycheck_Y.loc[ycheck_bottom, "unique_id"].nunique()
ycheck_bleft  = ycheck_Y["unique_id"].str.contains(r"/B\d{5,6}$").sum()
ycheck_retY   = int(ycheck_Y["unique_id"].str.startswith("Returnable").sum())   # Returnable rows left in Y_product
ycheck_first  = ycheck_Y["ds"].min()
ycheck_last   = ycheck_Y["ds"].max()
ycheck_weeks  = ycheck_Y["ds"].nunique()
ycheck_ndimu  = ycheck_Y.loc[(ycheck_Y["unique_id"] == "Beer/Mainstream/Tusker Ndimu/782264")
                             & (ycheck_Y["ds"] == "2025-08-25"), "y"].round(2).tolist()

print("Y_product rows:", ycheck_rows, "(expect 17022)")
print("Series:", ycheck_series, "(expect 282)")
print("Products:", ycheck_prods, "(expect 188)")
print("B-code products left:", ycheck_bleft, "(expect 0)")
print("Returnable rows left in Y_product:", ycheck_retY, "(expect 0)")
print("S_product shape:", ycheck_S.shape, "(expect (282, 189) = unique_id + 188 products)")
print("Weeks:", ycheck_first, "to", ycheck_last, f"({ycheck_weeks} weeks)", "(expect 2025-04-07 to 2026-07-06, 66)")
print("Tusker Ndimu 782264, week of 25 Aug 2025:", ycheck_ndimu, "(expect [0.84])")
print("Y_returnable:", ycheck_R.shape, "|", ycheck_R["unique_id"].nunique(), "items", "(expect (926, 3) | 15 items)")

assert ycheck_series == 282 and ycheck_prods == 188, "series/products wrong - re-run the Prep notebook (Restart + Run All)"
assert ycheck_bleft == 0, "B-codes in Y_product - re-run Cleaning cell 53 (C1), then Prep"
assert ycheck_retY == 0, "Returnable still in Y_product - check the Prep R1 cell (after cell 9), then Restart + Run All"
assert ycheck_S.shape == (282, 189), "S_product shape wrong - re-run Prep"
assert ycheck_R["unique_id"].nunique() == 15 and len(ycheck_R) == 926, "Y_returnable wrong - re-run Prep (R1 cell)"
assert (ycheck_first, ycheck_last, ycheck_weeks) == ("2025-04-07", "2026-07-06", 66), \
    "dates wrong - re-run Cleaning D1 cell (54), then Prep"
# ---- Per-series checks: weeks in order, no gaps, no duplicates (each of the 282 series) ----
ycheck_Y["ycheck_d"] = pd.to_datetime(ycheck_Y["ds"])                     # ds is read from the CSV as text
ycheck_g          = ycheck_Y.groupby("unique_id", sort=False)["ycheck_d"]  # one group per series
ycheck_unordered  = int((~ycheck_g.apply(lambda d: d.is_monotonic_increasing)).sum())
ycheck_gappy      = int(ycheck_g.apply(lambda d: (d.diff().dropna() != pd.Timedelta(days=7)).any()).sum())
ycheck_dups       = int(ycheck_Y.duplicated(["unique_id", "ds"]).sum())
ycheck_notmon     = int((ycheck_Y["ycheck_d"].dt.dayofweek != 0).sum())   # 0 = Monday
ycheck_badend     = int((ycheck_g.max() != ycheck_Y["ycheck_d"].max()).sum())
print(f"Series out of order: {ycheck_unordered} | with gaps: {ycheck_gappy} | duplicate weeks: {ycheck_dups} | "
      f"non-Monday weeks: {ycheck_notmon} | not ending on last week: {ycheck_badend}   (expect all 0)")
assert ycheck_unordered == 0 and ycheck_gappy == 0 and ycheck_dups == 0, \
    "a series has weeks out of order, a gap or a duplicate - re-run Prep (Restart + Run All)"
assert ycheck_notmon == 0 and ycheck_badend == 0, "week labels wrong - re-run Prep (Restart + Run All)"
print("\nAll checks PASS")


### ==========================================================================
## DEMO ONLY - a small made-up table shaped like df. Shows what "merge each split pair
## into one line" would do. It does NOT touch df or any file.
### ==========================================================================

In [ ]:

demo = pd.DataFrame({
    "source_file":     ["DailySalesReportNew_0101202615012026.xlsx"] * 7,
    "salesperson":     ["VINCENT MUSERA-OLK ROUTE"] * 7,
    "customer_code":   ["KE0083739"] * 7,
    "segment":         ["Bar"] * 7,
    "visit_date":      pd.to_datetime(["2026-01-13"] * 7),
    "product":         ["Euroform Amber Local 500ml", "Guinness (K) Per Bottle", "Complete - Guiness Genius",
                        "Euroform Amber Local 500ml", "Complete - Guiness Genius", "White Cap Local 500ml 25X01",
                        "Mis Plastic Container Ever"],
    "Product_Code":    ["350600", "696894", "350360", "350600", "350360", "616861", "350593"],
    "Category":        ["Returnable", "Beer", "Returnable", "Returnable", "Returnable", "Beer", "Returnable"],
    "quantity":        [0, 0.40, 0, 25, 2, 2, 1],
    "return_quantity": [25, 0, 2, 0, 0, 0, 0],
    "Cases_Conversion":[0.04, 1.0, 1.0, 0.04, 1.0, 1.0, 1.0],
})
demo["quantity_cases"] = demo["quantity"] * demo["Cases_Conversion"]
demo["returns_cases"]  = demo["return_quantity"] * demo["Cases_Conversion"]

print("BEFORE - 7 lines (the split halves are lines 0+3 and 2+4):")
display(demo[["visit_date", "customer_code", "product", "Category", "quantity", "return_quantity"]])

# ---- The merge ----
key = ["customer_code", "visit_date", "product"]
ret_half  = demo[(demo["quantity"] == 0) & (demo["return_quantity"] > 0)]
sale_half = demo[(demo["quantity"] > 0) & (demo["return_quantity"] == 0)]

# pair each return half with a sale half: same customer, date, product AND same number
ret_keys  = ret_half.assign(ret_row=ret_half.index,  n=ret_half["return_quantity"].astype(float))[key + ["n", "ret_row", "return_quantity"]]
sale_keys = sale_half.assign(sale_row=sale_half.index, n=sale_half["quantity"].astype(float))[key + ["n", "sale_row"]]
matched = ret_keys.merge(sale_keys, on=key + ["n"]).drop_duplicates("ret_row").drop_duplicates("sale_row")

merged = demo.copy()
merged.loc[matched["sale_row"], "return_quantity"] = matched["return_quantity"].values   # move the return onto the sale line
merged["returns_cases"] = merged["return_quantity"] * merged["Cases_Conversion"]
merged = merged.drop(index=matched["ret_row"])                                            # drop the now-empty return half

print(f"\nAFTER - {len(merged)} lines ({len(matched)} pairs combined):")
display(merged[["visit_date", "customer_code", "product", "Category", "quantity", "return_quantity"]])

# ---- Proof nothing was lost ----
check = pd.DataFrame({
    "Before": [demo["quantity_cases"].sum(),   demo["return_quantity"].sum(),   len(demo)],
    "After":  [merged["quantity_cases"].sum(), merged["return_quantity"].sum(), len(merged)],
}, index=["Total quantity (cases)", "Total returns (units)", "Number of lines"])
print("\nTotals before vs after:")
display(check)
